# CheckKro 
**Know if your data is safe to analyze.**

CheckKro is a tool that checks any CSV file the way an analyst would on first opening it, and answers one question: *can I trust this data?*

### What it does
1. **Checks** the file for the problems analysts look for first: missing and hidden-missing values, duplicates, formatting mistakes, impossible values, outliers, date errors and suspicious patterns.
2. **Scores** the file out of 100 and gives a verdict: safe to analyze, use with caution, or do not use until fixed.
3. **Explains the impact** of each problem on your numbers, for example "total revenue is overstated by 3.1%".
4. **Cleans safely**: trims spaces, turns placeholders like "N/A" into real blanks, and removes exact duplicates, with every change logged. Risky fixes are left to a human.

### How this notebook is organised
Test data, then the checks, then the score, the cleaner, the impact feature, the Streamlit app, and finally the tests that measure how well it works.

*The checks are plain rule-based Python, not AI, so every result is repeatable and explainable.*

## 1. Test data

Before building any checks, we need a file where we **know exactly what is wrong**. Otherwise we can't prove the checks work.

This cell writes `datagen.py`, which has two functions:
- `make_clean()` builds a healthy table of 5,000 delivery orders (ages, amounts, ratings, cities, dates and refund flags).
- `make_messy()` copies that table and breaks it on purpose.

| Planted problem | Column | Amount |
|---|---|---|
| Missing values | `rating` | 8% |
| Hidden missing values ("N/A") | `city` | 5% |
| Inconsistent casing ("blinkit") | `platform` | 5% |
| Impossible age (150) | `customer_age` | 1% |
| Delivered before ordered | `delivery_date` | 2% |
| Implausible refund rate | `refund_flag` | about 42% |
| Duplicate rows | all columns | 3% |

Each check we build later is tested against one of these known problems.

In [1]:
%%writefile datagen.py
import numpy as np
import pandas as pd

def make_clean(n=5000, seed=0):
    r = np.random.default_rng(seed)
    order_date = pd.Timestamp("2024-01-01") + pd.to_timedelta(r.integers(0, 330, n), unit="D")
    return pd.DataFrame({
        "order_id": [f"ORD{i:06d}" for i in range(n)],
        "customer_age": np.clip(r.normal(36, 11, n).round(), 18, 75).astype(int),
        "order_amount": r.lognormal(6, 0.35, n).round(2),
        "rating": r.choice([1, 2, 3, 4, 5], n, p=[.05, .07, .18, .4, .3]),
        "city": r.choice(["Delhi", "Mumbai", "Bengaluru", "Pune", "Chennai"], n),
        "platform": r.choice(["Blinkit", "Zepto", "Instamart"], n),
        "order_date": order_date,
        "delivery_date": order_date + pd.to_timedelta(r.integers(0, 4, n), unit="D"),
        "refund_flag": (r.random(n) < 0.04).astype(int),
    })

def make_messy(n=5000, seed=0):
    r = np.random.default_rng(seed)
    df = make_clean(n, seed)
    pick = lambda frac: r.choice(n, int(n * frac), replace=False)

    df["rating"] = df["rating"].astype(float)
    df.loc[pick(0.08), "rating"] = np.nan            # real missing values
    df["city"] = df["city"].astype(object)
    df.loc[pick(0.05), "city"] = "N/A"               # disguised missing
    df.loc[pick(0.05), "platform"] = "blinkit"       # inconsistent casing
    df.loc[pick(0.01), "customer_age"] = 150         # impossible age
    df.loc[pick(0.02), "delivery_date"] = df["order_date"] - pd.Timedelta(days=3)  # delivered before ordered
    df.loc[pick(0.40), "refund_flag"] = 1            # implausible refund rate
    df = pd.concat([df, df.sample(frac=0.03, random_state=seed)], ignore_index=True)  # duplicate rows
    return df

if __name__ == "__main__":
    make_messy().to_csv("messy_orders.csv", index=False)
    print("Saved messy_orders.csv")

Overwriting datagen.py


**Generate the messy file.** This loads the test data into a table called `df` and saves a copy as `messy_orders.csv`. The shape `(5150, 9)` means 5,150 rows and 9 columns. The extra 150 rows beyond the original 5,000 are the duplicates we planted.

In [2]:
from datagen import make_messy
df = make_messy()
df.to_csv("messy_orders.csv", index=False)
print(df.shape)

(5150, 9)


**Look at the data the normal way.** This is the usual first check: count the blank values in each column. Only `rating` shows blanks (408). `city` shows 0, even though about 5% of its values are the text "N/A". Normal missing-value counts can't see them, and that is the gap our tool is built to close.

In [3]:
print(df.isna().sum())

order_id           0
customer_age       0
order_amount       0
rating           408
city               0
platform           0
order_date         0
delivery_date      0
refund_flag        0
dtype: int64


**Find the hidden blanks.** Counting the values in `city` shows "N/A" sitting among the real cities, 259 times. Python treats it as a normal city name, which is why the previous count missed it.

In [4]:
print(df["city"].value_counts())

city
Pune         1034
Bengaluru    1005
Mumbai        980
Chennai       946
Delhi         926
N/A           259
Name: count, dtype: int64


## 2. The Finding structure and the missing-values check

This cell creates `checks.py`, the file that holds every check in the tool.

First, it defines a `Finding`, the one shape every check reports in: which check, which column, how severe, how many rows, what was found, why it matters, and what to do. Because all checks speak the same language, the score, the cleaner and the app can all read the same list.

Then it adds the first check, for missing values, which looks for three things:
- Real blanks, the ones Python already counts.
- Hidden blanks: text like "N/A", "-", "null" or Excel's "#NAME?" that stands in for missing data.
- Blanks that follow a rule. If a column is empty only when another column has a certain value (for example, no rating when a ride was cancelled), it is reported as low severity, because it is probably by design and not an error.

In [5]:
%%writefile checks.py
from dataclasses import dataclass
import pandas as pd

# Text values that really mean "missing" (including Excel error values)
MISSING_TOKENS = {"", "na", "n/a", "nan", "null", "none", "-", "--", "?", "unknown", "missing", "#n/a",
                  "#name?", "#ref!", "#div/0!", "#value!", "#num!", "#null!"}


@dataclass
class Finding:
    check: str        # which check found it
    column: str       # which column
    severity: str     # low / medium / high
    count: int        # how many rows affected
    pct: float        # percent of all rows
    detail: str       # what exactly was found
    why: str          # why it matters
    suggestion: str   # what to do


def is_text(s):
    return pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)


def severity_from_pct(pct, high=20, medium=5):
    if pct >= high:
        return "high"
    if pct >= medium:
        return "medium"
    return "low"


def explained_by(df, col):
    """If the blanks in `col` follow a clean pattern in a small category column,
    return (that column, the values where `col` is blank). Otherwise None."""
    miss = df[col].isna()
    for other in df.columns:
        s = df[other]
        if other == col or not is_text(s) or not (2 <= s.nunique() <= 10):
            continue
        rate = miss.groupby(s).mean()
        if ((rate >= 0.95) | (rate <= 0.05)).all() and (rate >= 0.95).any() and (rate <= 0.05).any():
            return other, [str(v) for v in rate.index[rate >= 0.95]]
    return None


def check_missing(df):
    findings = []
    n = len(df)
    for col in df.columns:
        s = df[col]

        # 1. Real missing values (NaN)
        count = int(s.isna().sum())
        if count > 0:
            pct = round(count / n * 100, 2)
            why = explained_by(df, col) if pct >= 5 else None
            if why:
                other, vals = why
                findings.append(Finding(
                    check="structural_missing",
                    column=col,
                    severity="low",
                    count=count,
                    pct=pct,
                    detail=f"{pct}% of '{col}' is blank, but the pattern is explained by '{other}' (blank when it is: {', '.join(vals[:3])}). Probably by design, not an error.",
                    why="Blanks that follow a clear rule are usually expected, but they still limit what you can analyze.",
                    suggestion=f"Confirm the rule with the data owner, and analyze '{col}' only where it applies.",
                ))
            else:
                findings.append(Finding(
                    check="missing_values",
                    column=col,
                    severity=severity_from_pct(pct),
                    count=count,
                    pct=pct,
                    detail=f"{count} empty values ({pct}%) in '{col}'.",
                    why="Averages, counts and models can be skewed if missing rows are ignored.",
                    suggestion="Decide per column: fill (median/mode), drop rows, or leave and report it.",
                ))

        # 2. Hidden missing values (text like "N/A" or Excel's "#NAME?")
        if is_text(s):
            cleaned = s.dropna().astype(str).str.strip().str.lower()
            hidden = int(cleaned.isin(MISSING_TOKENS).sum())
            if hidden > 0:
                pct = round(hidden / n * 100, 2)
                findings.append(Finding(
                    check="disguised_missing",
                    column=col,
                    severity=severity_from_pct(pct),
                    count=hidden,
                    pct=pct,
                    detail=f"{hidden} values ({pct}%) in '{col}' look like placeholders for missing data (e.g. 'N/A' or '#NAME?').",
                    why="Normal missing-value checks miss these, so they get counted as real categories.",
                    suggestion="Convert these to real missing values before analysis.",
                ))
    return findings

Overwriting checks.py


**Run the missing-values check.** This runs the check on our messy file and prints what it found. It reports two problems: the empty values in `rating` (about 8%), and the hidden "N/A" values in `city` (about 5%) that a normal blank count could not see.

In [6]:
from checks import check_missing

for f in check_missing(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

MEDIUM | missing_values | rating
   408 empty values (7.92%) in 'rating'.
MEDIUM | disguised_missing | city
   259 values (5.03%) in 'city' look like placeholders for missing data (e.g. 'N/A' or '#NAME?').


## 3. Duplicates

This cell adds the duplicates check to `checks.py`. The `-a` means "append", so it adds to the file instead of replacing it.

It looks for two different problems:
- **Exact duplicate rows**, where every column matches another row. These inflate counts and totals.
- **Repeated IDs**, where the same ID appears with different data, which usually means a conflicting or mis-keyed record.

IDs named after a customer, user, driver and similar (like `customer_id`) are skipped on purpose, because one customer legitimately has many orders. An ID is also only flagged when a small share of it repeats, since an ID that repeats a lot is a shared key, not an error.

In [7]:
%%writefile -a checks.py


import re

# ID columns named after these things are expected to repeat (one customer, many orders)
FOREIGN_KEY_WORDS = {"customer", "user", "driver", "client", "account", "member", "product",
                     "store", "vendor", "supplier", "session", "device", "employee", "patient", "student"}


def check_duplicates(df):
    findings = []
    n = len(df)

    # 1. Exact duplicate rows (every column identical)
    count = int(df.duplicated().sum())
    if count > 0:
        pct = round(count / n * 100, 2)
        findings.append(Finding(
            check="exact_duplicates",
            column="(all columns)",
            severity=severity_from_pct(pct, high=5, medium=1),
            count=count,
            pct=pct,
            detail=f"{count} rows ({pct}%) are exact copies of another row.",
            why="Duplicates inflate counts, revenue and averages.",
            suggestion="Remove exact duplicate rows, keeping the first copy.",
        ))

    # 2. Repeated IDs (same ID, but the rest of the row is different)
    unique_rows = df[~df.duplicated()]
    for col in df.columns:
        tokens = set(re.split(r"[^a-z0-9]+", str(col).lower()))
        if "id" in tokens and not (tokens & FOREIGN_KEY_WORDS):
            repeated = int(unique_rows[col].duplicated().sum())
            if 0 < repeated <= 0.05 * n:
                pct = round(repeated / n * 100, 2)
                findings.append(Finding(
                    check="duplicate_ids",
                    column=col,
                    severity=severity_from_pct(pct, high=5, medium=1),
                    count=repeated,
                    pct=pct,
                    detail=f"{repeated} rows ({pct}%) reuse an ID in '{col}' that already exists, with different data.",
                    why="An ID should identify one record. Repeats mean conflicting or mis-keyed data.",
                    suggestion="Investigate which version is correct before removing anything.",
                ))
    return findings

Appending to checks.py


**Run the duplicates check.** The `importlib.reload` line makes Python pick up the latest version of `checks.py`, since the file just changed. The check finds 150 exact duplicate rows, about 2.9% of the file, which are the copies we planted on purpose.

In [8]:
import importlib
import checks
importlib.reload(checks)

for f in checks.check_duplicates(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

MEDIUM | exact_duplicates | (all columns)
   150 rows (2.91%) are exact copies of another row.


## 4. Formatting problems

This cell adds the formatting check to `checks.py`. It only looks at text columns, and finds three kinds of problem:
- **Extra spaces** at the start or end of a value. "Delhi" and "Delhi " look identical but count as two different cities.
- **Inconsistent casing**, where the same word is written in different ways, like "Blinkit" and "blinkit". Any total or group-by splits them into separate categories.
- **Numbers stored as text.** If a column is entirely numbers but saved as text, it is reported as a low-severity type problem. If it is mostly numbers with a few text values mixed in (like "abc" in an amount column), it is reported as mixed types, because those few values stop the whole column from working as a number.

In [9]:
%%writefile -a checks.py


def check_formatting(df):
    findings = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        if not is_text(s):
            continue
        values = s.dropna().astype(str)
        if len(values) == 0:
            continue

        # 1. Extra spaces at the start or end
        stripped = values.str.strip()
        spaces = int((values != stripped).sum())
        if spaces > 0:
            pct = round(spaces / n * 100, 2)
            findings.append(Finding(
                check="extra_whitespace",
                column=col,
                severity=severity_from_pct(pct, high=20, medium=5),
                count=spaces,
                pct=pct,
                detail=f"{spaces} values ({pct}%) in '{col}' have spaces at the start or end.",
                why="'Delhi' and 'Delhi ' look the same but group as different categories.",
                suggestion="Trim the spaces.",
            ))

        # 2. Inconsistent casing (same word, different capitalisation)
        if stripped.nunique() / len(stripped) < 0.5:
            counts = stripped.value_counts()
            table = pd.DataFrame({"raw": counts.index, "n": counts.values})
            table["key"] = table["raw"].str.lower()
            mixed = table.groupby("key").filter(lambda g: len(g) > 1)
            if len(mixed) > 0:
                minority = int(mixed.groupby("key")["n"].apply(lambda x: x.sum() - x.max()).sum())
                pct = round(minority / n * 100, 2)
                examples = ", ".join(f"'{v}'" for v in list(mixed["raw"])[:4])
                findings.append(Finding(
                    check="inconsistent_case",
                    column=col,
                    severity=severity_from_pct(pct, high=20, medium=5),
                    count=minority,
                    pct=pct,
                    detail=f"'{col}' has the same value written in different cases ({examples}). About {minority} rows ({pct}%) use the less common spelling.",
                    why="Group-bys and filters treat each spelling as a separate category, splitting your totals.",
                    suggestion="Standardise to one spelling (e.g. the most common one).",
                ))

        # 3. Numbers stored as text
        present = stripped[~stripped.str.lower().isin(MISSING_TOKENS)]
        if len(present) > 0:
            num = pd.to_numeric(present.str.replace(",", "", regex=False), errors="coerce")
            ratio = num.notna().mean()
            if ratio == 1:
                findings.append(Finding(
                    check="numeric_as_text",
                    column=col,
                    severity="low",
                    count=len(present),
                    pct=round(len(present) / n * 100, 2),
                    detail=f"'{col}' holds only numbers but is stored as text.",
                    why="Sorting and maths on text columns give wrong results.",
                    suggestion="Convert the column to a numeric type.",
                ))
            elif ratio >= 0.8:
                bad = int(num.isna().sum())
                pct = round(bad / n * 100, 2)
                findings.append(Finding(
                    check="mixed_types",
                    column=col,
                    severity=severity_from_pct(pct, high=5, medium=1),
                    count=bad,
                    pct=pct,
                    detail=f"'{col}' is mostly numbers, but {bad} values ({pct}%) are text.",
                    why="A few text values stop the whole column from being used as a number.",
                    suggestion="Find the text values and correct or remove them.",
                ))
    return findings

Appending to checks.py


**Run the formatting check.** It finds one problem: `platform` has "Blinkit" and "blinkit", with about 262 rows (5%) using the less common lowercase spelling. Nothing is reported for the other columns, which is the right result, because we only planted that one formatting problem.

In [10]:
import importlib
import checks
importlib.reload(checks)

for f in checks.check_formatting(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

MEDIUM | inconsistent_case | platform
   'platform' has the same value written in different cases ('Blinkit', 'blinkit'). About 262 rows (5.09%) use the less common spelling.


## 5. Impossible values and outliers

This cell adds the values check to `checks.py`. It only looks at number columns (and skips IDs), and finds three kinds of problem:
- **Negative values** in columns that can never be negative, such as age, price, amount or quantity. The check decides this from the column name.
- **Impossible ages**, anything above 120.
- **Extreme outliers**, values far outside the normal range. The limit is five times the middle spread of the data, so only truly extreme values are flagged, not ordinary highs and lows.

Impossible values are always rated high severity, because they are almost certainly entry errors. Outliers are rated by how many rows they affect, since an extreme value can be real.

In [11]:
%%writefile -a checks.py


# Column-name words that tell us a value can never be negative
NON_NEGATIVE_WORDS = {"age", "price", "amount", "quantity", "qty", "count",
                      "total", "fee", "cost", "salary", "distance", "duration"}


def tokens_of(col):
    return set(re.split(r"[^a-z0-9]+", str(col).lower()))


def check_values(df):
    findings = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        if not pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            continue
        toks = tokens_of(col)
        if "id" in toks:
            continue

        # 1. Negative values where they make no sense
        if toks & NON_NEGATIVE_WORDS:
            neg = int((s < 0).sum())
            if neg > 0:
                pct = round(neg / n * 100, 2)
                findings.append(Finding(
                    check="impossible_values",
                    column=col,
                    severity="high",
                    count=neg,
                    pct=pct,
                    detail=f"{neg} rows ({pct}%) in '{col}' are negative, which should not be possible.",
                    why="Negative values usually mean entry errors or refunds mixed into sales data.",
                    suggestion="Check the source. Correct, separate or remove these rows.",
                ))

        # 2. Ages above 120
        if "age" in toks:
            too_old = int((s > 120).sum())
            if too_old > 0:
                pct = round(too_old / n * 100, 2)
                findings.append(Finding(
                    check="impossible_values",
                    column=col,
                    severity="high",
                    count=too_old,
                    pct=pct,
                    detail=f"{too_old} rows ({pct}%) in '{col}' are above 120, which is not a realistic age.",
                    why="Impossible ages distort averages and age-group analysis.",
                    suggestion="Treat as errors. Correct from source or set to missing.",
                ))

        # 3. Extreme outliers (far outside the normal range)
        if s.nunique() > 20:
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
            iqr = q3 - q1
            if iqr > 0:
                low, high = q1 - 5 * iqr, q3 + 5 * iqr
                out = int(((s < low) | (s > high)).sum())
                if out > 0:
                    pct = round(out / n * 100, 2)
                    findings.append(Finding(
                        check="extreme_outliers",
                        column=col,
                        severity=severity_from_pct(pct, high=5, medium=1),
                        count=out,
                        pct=pct,
                        detail=f"{out} rows ({pct}%) in '{col}' fall outside {low:.1f} to {high:.1f}, far beyond the normal range.",
                        why="A few extreme values can drag averages and totals far from reality.",
                        suggestion="Check whether they are real. Keep, cap, or remove with a note.",
                    ))
    return findings

Appending to checks.py


**Run the values check.** It finds the planted ages of 150 in `customer_age` (53 rows, about 1%), reported twice: once as an impossible value, and once as an extreme outlier. It also flags a single `order_amount` row (0.02%) that landed far out in the natural spread of the random data, rated low severity, so it reads as "take a look" and not as an error.

In [12]:
import importlib
import checks
importlib.reload(checks)

for f in checks.check_values(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

HIGH | impossible_values | customer_age
   53 rows (1.03%) in 'customer_age' are above 120, which is not a realistic age.
MEDIUM | extreme_outliers | customer_age
   53 rows (1.03%) in 'customer_age' fall outside -46.0 to 119.0, far beyond the normal range.
LOW | extreme_outliers | order_amount
   1 rows (0.02%) in 'order_amount' fall outside -657.1 to 1496.7, far beyond the normal range.


## 6. Dates and suspicious patterns

This cell adds the last big check to `checks.py`. It first finds the date columns, including dates stored as plain text, and then looks for three problems:
- **Future dates** in data that should be historical. Columns that are normally in the future, like expiry or due dates, are skipped.
- **Events in the wrong order**, such as a delivery dated before its order. This breaks any time-based measure, like delivery time.
- **Suspiciously high rates** in yes/no flag columns such as refunds, cancellations and returns. A refund rate above 25% is flagged, and above 40% is rated high severity. This is the check inspired by the grocery-data project, where a 46% refund rate gave away that the data was synthetic.

In [13]:
%%writefile -a checks.py


import warnings

# Pairs of (earlier event, later event) that must happen in this order
DATE_PAIRS = [("order", "deliver"), ("order", "ship"), ("start", "end"), ("created", "updated")]
# Date columns that are normally in the future, so we skip them
FUTURE_OK = ("expir", "due", "end", "renew", "next", "valid")
# Columns that are yes/no flags where a very high rate is suspicious
RATE_WORDS = {"refund", "cancel", "return", "fraud", "chargeback"}


def date_columns(df):
    out = {}
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            out[col] = s
        elif is_text(s):
            sample = s.dropna().astype(str).head(200)
            if len(sample) > 0 and sample.str.match(r"^\s*\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}").mean() >= 0.9:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    parsed = pd.to_datetime(s, errors="coerce")
                if parsed.notna().sum() >= 0.8 * s.notna().sum():
                    out[col] = parsed
    return out


def check_dates_and_patterns(df):
    findings = []
    n = len(df)
    dates = date_columns(df)

    # 1. Dates in the future
    now = pd.Timestamp.now()
    for col, d in dates.items():
        if any(w in col.lower() for w in FUTURE_OK):
            continue
        future = int((d > now).sum())
        if future > 0:
            pct = round(future / n * 100, 2)
            findings.append(Finding(
                check="future_dates", column=col, severity="high",
                count=future, pct=pct,
                detail=f"{future} rows ({pct}%) in '{col}' are dated in the future.",
                why="Events that have not happened yet cannot be in historical data.",
                suggestion="Check for typos in the year or a wrong date format.",
            ))

    # 2. Events happening in the wrong order
    for first, second in DATE_PAIRS:
        for a in [c for c in dates if first in c.lower()]:
            for b in [c for c in dates if second in c.lower() and c != a]:
                both = dates[a].notna() & dates[b].notna()
                bad = int((dates[a][both] > dates[b][both]).sum())
                if bad > 0:
                    pct = round(bad / n * 100, 2)
                    findings.append(Finding(
                        check="date_order", column=b, severity="high",
                        count=bad, pct=pct,
                        detail=f"{bad} rows ({pct}%) have '{b}' earlier than '{a}'.",
                        why="Time-based metrics (like delivery time) become negative or meaningless.",
                        suggestion="Check whether the two columns were swapped or mis-entered.",
                    ))

    # 3. Suspiciously high rates in yes/no flag columns
    for col in df.columns:
        if not (tokens_of(col) & RATE_WORDS):
            continue
        s = df[col].dropna()
        if len(s) == 0:
            continue
        if pd.api.types.is_numeric_dtype(s) and s.isin([0, 1]).all():
            rate = s.mean()
        elif is_text(s) and s.astype(str).str.strip().str.lower().isin(["yes", "no", "y", "n", "true", "false"]).all():
            rate = s.astype(str).str.strip().str.lower().isin(["yes", "y", "true"]).mean()
        else:
            continue
        if rate > 0.25:
            findings.append(Finding(
                check="suspicious_rate", column=col,
                severity="high" if rate >= 0.4 else "medium",
                count=int(round(rate * len(s))), pct=round(rate * 100, 2),
                detail=f"'{col}' is positive in {rate*100:.1f}% of rows, far higher than is normal for this kind of event (usually in the low single digits to low teens).",
                why="Rates this high often signal synthetic data, a logging bug, or a mislabeled column.",
                suggestion="Verify against the source system before drawing conclusions.",
            ))
    return findings

Appending to checks.py


**Run the dates and patterns check.** It finds both planted problems: 102 orders (about 2%) where `delivery_date` is earlier than `order_date`, and a `refund_flag` rate of about 42.6%, far above the normal range. Both are rated high severity. No future dates are reported, which is correct, because every date in the file is from 2024.

In [14]:
import importlib
import checks
importlib.reload(checks)

for f in checks.check_dates_and_patterns(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

HIGH | date_order | delivery_date
   102 rows (1.98%) have 'delivery_date' earlier than 'order_date'.
HIGH | suspicious_rate | refund_flag
   'refund_flag' is positive in 42.6% of rows, far higher than is normal for this kind of event (usually in the low single digits to low teens).


## 7. File overview and column profile

This cell adds two things to `checks.py`, both from the routine an analyst runs on every new file.

The **overview** is the first table an analyst prints: for each column, its type, the percentage missing (counting hidden blanks like "N/A"), the number of unique values, and an example value.

The **column profile check** looks for three low-severity oddities:
- A column with the same value in every row, which carries no information.
- A text column where every value is unique, which is probably an ID or free text and can't be used to group rows.
- Column names with spaces or special characters, which cause errors in code, SQL and BI tools.

In [15]:
%%writefile -a checks.py


def overview(df):
    """The table an analyst prints first: type, missing %, unique count per column."""
    rows = []
    for col in df.columns:
        s = df[col]
        hidden = 0
        if is_text(s):
            hidden = int(s.dropna().astype(str).str.strip().str.lower().isin(MISSING_TOKENS).sum())
        rows.append({
            "column": col,
            "type": str(s.dtype),
            "missing_%": round((s.isna().sum() + hidden) / len(df) * 100, 2),
            "unique": int(s.nunique()),
            "example": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(rows)


def check_column_profile(df):
    findings = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        present = s.dropna()

        # 1. Column with only one value
        if n > 1 and present.nunique() == 1:
            findings.append(Finding(
                check="constant_column", column=col, severity="low",
                count=n, pct=100.0,
                detail=f"'{col}' has the same value in every row.",
                why="A column that never changes carries no information for analysis.",
                suggestion="Drop it, or confirm it should really be constant.",
            ))

        # 2. Text column where every value is unique (looks like an ID or free text)
        if (is_text(s) and n > 20 and len(present) > 0
                and present.nunique() == len(present) and "id" not in tokens_of(col)):
            findings.append(Finding(
                check="all_unique_column", column=col, severity="low",
                count=len(present), pct=round(len(present) / n * 100, 2),
                detail=f"Every value in '{col}' is unique, so it is probably an identifier or free text.",
                why="It cannot be used to group or compare rows.",
                suggestion="Exclude it from group-by analysis, or confirm it is meant to be an ID.",
            ))

        # 3. Messy column name
        name = str(col)
        if name != name.strip() or re.search(r"[^A-Za-z0-9_]", name.strip()):
            findings.append(Finding(
                check="messy_column_name", column=col, severity="low",
                count=0, pct=0.0,
                detail=f"Column name '{col}' has spaces or special characters.",
                why="Awkward names cause errors in code, SQL and BI tools.",
                suggestion="Rename to lowercase_with_underscores.",
            ))
    return findings

Appending to checks.py


**Run the overview and column profile.** The table shows each column's type, percent missing, unique count and an example. Note that `city` shows about 5% missing here, even though the normal blank count said 0, because the overview also counts the hidden "N/A" values. No column-profile problems are reported, which is the right result for this file: it has no constant columns, no all-unique text columns, and clean column names.

In [16]:
import importlib
import checks
importlib.reload(checks)

print(checks.overview(df))
print()
for f in checks.check_column_profile(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

          column            type  missing_%  unique              example
0       order_id          object       0.00    5000            ORD000000
1   customer_age           int64       0.00      55                   54
2   order_amount         float64       0.00    4783               341.53
3         rating         float64       7.92       5                  5.0
4           city          object       5.03       6                 Pune
5       platform          object       0.00       4            Instamart
6     order_date  datetime64[ns]       0.00     330  2024-10-07 00:00:00
7  delivery_date  datetime64[ns]       0.00     334  2024-10-07 00:00:00
8    refund_flag           int64       0.00       2                    0



## 8. Date coverage and rare categories

This cell adds three more first-look items to `checks.py`.

**Date coverage** shows, for each date column, the earliest date, the latest date, and how many distinct days appear in between. It tells you at a glance whether the file covers the period you expect.

**Date gaps** checks daily data for missing stretches. If whole days are absent, trends and week-on-week comparisons quietly break. It only runs on data that is recorded daily, because gaps are normal in weekly or monthly data.

**Rare categories** looks for values that appear only once or twice in a category column, such as "Dehli" next to "Delhi". Those are often typos.

In [17]:
%%writefile -a checks.py


def date_coverage(df):
    """Earliest date, latest date and number of distinct days for each date column."""
    rows = []
    for col, d in date_columns(df).items():
        days = d.dropna().dt.normalize()
        if len(days) == 0:
            continue
        rows.append({
            "column": col,
            "earliest": days.min().date(),
            "latest": days.max().date(),
            "span_days": (days.max() - days.min()).days + 1,
            "distinct_days": int(days.nunique()),
        })
    return pd.DataFrame(rows)


def check_date_gaps(df):
    findings = []
    for col, d in date_columns(df).items():
        days = d.dropna().dt.normalize().drop_duplicates().sort_values()
        if len(days) < 30:
            continue
        diffs = days.diff().dt.days.dropna()
        if diffs.median() > 1:          # not daily data, so gaps are normal
            continue
        big = diffs[diffs > 3]
        if len(big) == 0:
            continue
        span = (days.max() - days.min()).days + 1
        missing = int((big - 1).sum())
        pct = round(missing / span * 100, 2)
        worst = diffs.idxmax()
        start = days.shift(1).loc[worst].date()
        end = days.loc[worst].date()
        findings.append(Finding(
            check="date_gaps", column=col,
            severity=severity_from_pct(pct, high=10, medium=2),
            count=missing, pct=pct,
            detail=f"'{col}' has {missing} missing days ({pct}% of its range). Longest gap: {start} to {end}.",
            why="Gaps in a daily series quietly break trends, averages and week-on-week comparisons.",
            suggestion="Check whether the data was not collected or was lost in an extract.",
        ))
    return findings


def check_rare_categories(df):
    findings = []
    n = len(df)
    if n < 100:
        return findings
    for col in df.columns:
        s = df[col]
        if not is_text(s):
            continue
        values = s.dropna().astype(str).str.strip()
        if values.nunique() > 50 or values.nunique() / max(len(values), 1) >= 0.5:
            continue
        counts = values.value_counts()
        rare = [v for v, c in counts.items() if c <= 2 and v.lower() not in MISSING_TOKENS]
        if rare:
            examples = ", ".join(f"'{v}'" for v in rare[:5])
            findings.append(Finding(
                check="rare_categories", column=col, severity="low",
                count=len(rare), pct=round(len(rare) / n * 100, 2),
                detail=f"'{col}' has {len(rare)} values that appear only once or twice ({examples}).",
                why="Very rare category values are often typos or entry mistakes.",
                suggestion="Check whether they are real categories or misspellings of a common one.",
            ))
    return findings

Appending to checks.py


**Run date coverage, date gaps and rare categories.** The coverage table shows `order_date` running from January to late November 2024, and `delivery_date` starting a day earlier, on 2023-12-31. That earlier start is the planted "delivered before ordered" error showing up in the date range. No gaps or rare categories are reported, which is correct: the dates cover almost every day, and every city appears hundreds of times.

In [18]:
import importlib
import checks
importlib.reload(checks)

print(checks.date_coverage(df))
print()
for f in checks.check_date_gaps(df) + checks.check_rare_categories(df):
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("  ", f.detail)

          column    earliest      latest  span_days  distinct_days
0     order_date  2024-01-01  2024-11-25        330            330
1  delivery_date  2023-12-31  2024-11-28        334            334



## 9. The runner

Until now, each check was run by hand. This cell adds `run_checkup`, one function that runs every check and returns a single list of findings, with the most serious first (high, then medium, then low, and within each level the ones affecting the most rows).

Everything built from here on uses this one function: the score, the cleaner, the impact sentences and the app. To add a new check later, you only write the check and add it to the `ALL_CHECKS` list.

In [19]:
%%writefile -a checks.py


ALL_CHECKS = [
    check_missing,
    check_duplicates,
    check_formatting,
    check_values,
    check_dates_and_patterns,
    check_column_profile,
    check_date_gaps,
    check_rare_categories,
]

SEVERITY_ORDER = {"high": 0, "medium": 1, "low": 2}


def run_checkup(df):
    """Run every check and return all findings, most serious first."""
    findings = []
    for check in ALL_CHECKS:
        findings.extend(check(df))
    findings.sort(key=lambda f: (SEVERITY_ORDER[f.severity], -f.pct))
    return findings

Appending to checks.py


**Run the full checkup.** All checks now run in one go on the messy file. It reports 9 findings, sorted from most to least serious: 3 high (the 42% refund rate, delivery before order, impossible ages), 5 medium and 1 low. Every problem we planted is found, and nothing unexpected shows up.

In [20]:
import importlib
import checks
importlib.reload(checks)

findings = checks.run_checkup(df)
print(len(findings), "findings\n")
for f in findings:
    print(f.severity.upper(), "|", f.check, "|", f.column, "|", f.pct, "%")

9 findings

HIGH | suspicious_rate | refund_flag | 42.56 %
HIGH | date_order | delivery_date | 1.98 %
HIGH | impossible_values | customer_age | 1.03 %
MEDIUM | missing_values | rating | 7.92 %
MEDIUM | inconsistent_case | platform | 5.09 %
MEDIUM | disguised_missing | city | 5.03 %
MEDIUM | exact_duplicates | (all columns) | 2.91 %
MEDIUM | extreme_outliers | customer_age | 1.03 %
LOW | extreme_outliers | order_amount | 0.02 %


## 10. Trust score and verdict

This cell creates `scoring.py`, a new file, so it does not use `-a`. It turns the list of findings into one number and one verdict.

The score starts at 100. Each finding takes points off: 12 for a high-severity finding, 5 for medium and 1 for low. The score never goes below 0.

The verdict follows from the score:
- **Safe to analyze**: the score is 85 or more and there are no high-severity findings.
- **Use with caution**: the score is 60 or more.
- **Do not use until fixed**: anything below that.

A single high-severity finding always blocks "safe to analyze", however high the score is, because one serious problem like an impossible value can make the whole file unreliable.

In [21]:
%%writefile scoring.py

PENALTY = {"high": 12, "medium": 5, "low": 1}


def trust_score(findings):
    """Start at 100 and subtract points for each finding."""
    score = 100 - sum(PENALTY[f.severity] for f in findings)
    return max(score, 0)


def verdict(score, findings):
    has_high = any(f.severity == "high" for f in findings)
    if score >= 85 and not has_high:
        return "SAFE TO ANALYZE"
    if score >= 60:
        return "USE WITH CAUTION"
    return "DO NOT USE UNTIL FIXED"

Overwriting scoring.py


**Run the score.** The messy file loses 62 points: 36 for the 3 high findings, 25 for the 5 medium ones and 1 for the low one. That leaves a trust score of 38 out of 100, and the verdict is "do not use until fixed".

In [22]:
import scoring
import importlib
importlib.reload(scoring)

score = scoring.trust_score(findings)
print("Trust score:", score, "/ 100")
print("Verdict:", scoring.verdict(score, findings))

Trust score: 38 / 100
Verdict: DO NOT USE UNTIL FIXED


## 11. The cleaner

This cell creates `cleaner.py`, another new file. It applies only the fixes that are safe to make without anyone's judgment, and records every one in a change log:
- Trim extra spaces in text columns.
- Turn placeholders like "N/A" or Excel's "#NAME?" into real blanks.
- Remove exact duplicate rows, keeping the first copy.

It returns two things: the cleaned table, and a log saying what was done, in which column and to how many rows. Nothing changes silently.

It deliberately does not fill in missing values, fix outliers or correct impossible ages. Those need a human decision, so they stay in the report as suggestions.

In [23]:
%%writefile cleaner.py
import numpy as np
import pandas as pd
from checks import MISSING_TOKENS, is_text


def clean(df):
    """Apply only safe fixes. Returns (cleaned_df, change_log)."""
    out = df.copy()
    log = []

    # 1. Trim extra spaces in text columns
    for col in out.columns:
        if not is_text(out[col]):
            continue
        s = out[col]
        trimmed = s.map(lambda v: v.strip() if isinstance(v, str) else v)
        changed = int(((s != trimmed) & s.notna()).sum())
        if changed > 0:
            out[col] = trimmed
            log.append({"action": "trimmed spaces", "column": col, "rows_affected": changed})

    # 2. Turn placeholders like "N/A" into real missing values
    for col in out.columns:
        if not is_text(out[col]):
            continue
        mask = out[col].map(lambda v: isinstance(v, str) and v.strip().lower() in MISSING_TOKENS)
        count = int(mask.sum())
        if count > 0:
            out.loc[mask, col] = np.nan
            log.append({"action": "placeholder -> missing", "column": col, "rows_affected": count})

    # 3. Remove exact duplicate rows (keep the first copy)
    before = len(out)
    out = out.drop_duplicates().reset_index(drop=True)
    removed = before - len(out)
    if removed > 0:
        log.append({"action": "removed duplicate rows", "column": "(all columns)", "rows_affected": removed})

    return out, pd.DataFrame(log, columns=["action", "column", "rows_affected"])

Overwriting cleaner.py


**Safety check.** This counts how many times each key function appears in `checks.py`. Each should appear exactly once. If a number were higher, a cell had been run twice and the code was added twice.

In [24]:
text = open("checks.py").read()
print("check_missing:", text.count("def check_missing"))
print("run_checkup:", text.count("def run_checkup"))

check_missing: 1
run_checkup: 1


**Run the cleaner.** The file goes from 5,150 rows to 5,000. The change log shows exactly two actions: 259 "N/A" placeholders in `city` turned into real blanks, and 150 duplicate rows removed. Nothing else was touched, and no spaces needed trimming because this file has none.

In [25]:
import cleaner
import importlib
importlib.reload(cleaner)

cleaned, log = cleaner.clean(df)
print("Rows before:", len(df), "| Rows after:", len(cleaned))
print()
print(log)

Rows before: 5150 | Rows after: 5000

                   action         column  rows_affected
0  placeholder -> missing           city            259
1  removed duplicate rows  (all columns)            150


**Check the cleaned file.** The same checkup runs again on the cleaned data. The score rises only from 38 to 43, and the verdict is still "do not use until fixed". That small rise is the point: the safe fixes remove the mechanical problems (duplicates, "N/A" placeholders), but the serious ones remain, such as the 42% refund rate, deliveries before orders and ages of 150. Those need a human decision, so the tool does not guess.

In [26]:
after = checks.run_checkup(cleaned)
after_score = scoring.trust_score(after)

print("BEFORE:", scoring.trust_score(findings), "/ 100 |", len(findings), "findings")
print("AFTER: ", after_score, "/ 100 |", len(after), "findings")
print("Verdict now:", scoring.verdict(after_score, after))
print()
for f in after:
    print(f.severity.upper(), "|", f.check, "|", f.column, "|", f.pct, "%")

BEFORE: 38 / 100 | 9 findings
AFTER:  43 / 100 | 8 findings
Verdict now: DO NOT USE UNTIL FIXED

HIGH | suspicious_rate | refund_flag | 42.58 %
HIGH | date_order | delivery_date | 2.0 %
HIGH | impossible_values | customer_age | 1.0 %
MEDIUM | missing_values | rating | 8.0 %
MEDIUM | missing_values | city | 5.0 %
MEDIUM | inconsistent_case | platform | 5.0 %
MEDIUM | extreme_outliers | customer_age | 1.0 %
LOW | extreme_outliers | order_amount | 0.02 %


## 12. Impact on your numbers

This cell creates `impact.py`, the feature that sets CheckKro apart. Most tools say "150 duplicate rows". This one says what that does to your results, in one plain sentence per problem:
- **Duplicates:** how much the total of money columns (like order amount) and the row count are overstated.
- **Missing values:** how little of the data any analysis by that column actually covers.
- **Impossible values and outliers:** how much the average of that column shifts with those rows compared to without them.
- **Delivery before order:** how the average delivery time changes with the bad rows removed.
- **Suspicious rates:** why any rate or trend built on that column cannot be trusted.
- **Inconsistent spelling:** how one category gets split into several smaller totals.

Money columns are found from their names (amount, price, total, revenue and so on). Findings with no clear effect on the numbers return nothing, so the app only shows an impact line where there is one.

In [27]:
%%writefile impact.py
import pandas as pd
from checks import tokens_of, date_columns, DATE_PAIRS

MONEY_WORDS = {"amount", "price", "total", "revenue", "cost", "fee", "sales"}


def money_columns(df):
    return [c for c in df.columns
            if pd.api.types.is_numeric_dtype(df[c]) and tokens_of(c) & MONEY_WORDS]


def impact(df, f):
    """One plain sentence on what a finding does to your numbers (or None)."""
    c, col = f.check, f.column

    if c == "exact_duplicates":
        dup = df[df.duplicated()]
        parts = []
        for m in money_columns(df):
            true_total = df[m].sum() - dup[m].sum()
            if true_total:
                parts.append(f"total '{m}' is overstated by {dup[m].sum() / true_total * 100:.1f}%")
        parts.append(f"the row count is overstated by {len(dup)}")
        return "Duplicates: " + " and ".join(parts) + "."

    if c in ("missing_values", "disguised_missing"):
        return (f"{f.pct}% of rows have no usable '{col}', so any analysis by '{col}' "
                f"covers only {100 - f.pct:.1f}% of the data.")

    if c in ("impossible_values", "extreme_outliers"):
        s = df[col]
        toks = tokens_of(col)
        if c == "impossible_values":
            bad = (s < 0) | ((s > 120) if "age" in toks else False)
        else:
            q1, q3 = s.quantile(0.25), s.quantile(0.75)
            iqr = q3 - q1
            bad = (s < q1 - 5 * iqr) | (s > q3 + 5 * iqr)
        if bad.sum() == 0 or (~bad).sum() == 0:
            return None
        with_bad, without = s.mean(), s[~bad].mean()
        shift = (with_bad - without) / without * 100 if without else 0
        return (f"The average '{col}' is {with_bad:.2f} with these rows and {without:.2f} without "
                f"({shift:+.1f}% shift).")

    if c == "date_order":
        dates = date_columns(df)
        for first, second in DATE_PAIRS:
            if second in col.lower():
                for a in [x for x in dates if first in x.lower() and x != col]:
                    gap = (dates[col] - dates[a]).dt.days
                    return (f"{f.pct}% of rows have a negative time between '{a}' and '{col}'. "
                            f"The average gap is {gap.mean():.2f} days with them and "
                            f"{gap[gap >= 0].mean():.2f} days without.")

    if c == "suspicious_rate":
        return (f"'{col}' reads {f.pct}% where the normal range is a few percent to the low teens, "
                f"so any rate, cost or trend built on it is unreliable.")

    if c == "inconsistent_case":
        return (f"Grouping by '{col}' splits one category into several, so each shows a smaller "
                f"total than it should ({f.pct}% of rows use the minority spelling).")

    return None

Overwriting impact.py


**Run the impact sentences.** Each of the 9 findings now comes with a plain sentence about its effect on the numbers. For example, the duplicates overstate total `order_amount` by 3.1% and the row count by 150. The impossible ages push the average `customer_age` up by 3.2%. The early deliveries drag the average delivery time down from 1.49 days to 1.40.

In [28]:
import impact
import importlib
importlib.reload(impact)

for f in findings:
    print(f.severity.upper(), "|", f.check, "|", f.column)
    print("   ->", impact.impact(df, f))
    print()

HIGH | suspicious_rate | refund_flag
   -> 'refund_flag' reads 42.56% where the normal range is a few percent to the low teens, so any rate, cost or trend built on it is unreliable.

HIGH | date_order | delivery_date
   -> 1.98% of rows have a negative time between 'order_date' and 'delivery_date'. The average gap is 1.40 days with them and 1.49 days without.

HIGH | impossible_values | customer_age
   -> The average 'customer_age' is 37.53 with these rows and 36.36 without (+3.2% shift).

MEDIUM | missing_values | rating
   -> 7.92% of rows have no usable 'rating', so any analysis by 'rating' covers only 92.1% of the data.

MEDIUM | inconsistent_case | platform
   -> Grouping by 'platform' splits one category into several, so each shows a smaller total than it should (5.09% of rows use the minority spelling).

MEDIUM | disguised_missing | city
   -> 5.03% of rows have no usable 'city', so any analysis by 'city' covers only 95.0% of the data.

MEDIUM | exact_duplicates | (all columns)


**Check that Streamlit is installed.** Streamlit is the library that turns the checks into a web page, where someone can upload a file and see the results. This prints its version, which confirms it is available before we build the app.

In [29]:
import streamlit
print("Streamlit version:", streamlit.__version__)

Streamlit version: 1.45.1


## 13. The web app

This cell creates `app.py`, which turns everything above into a web page. It does not run in the notebook. It is started from the terminal with `streamlit run app.py`.

The page works like this:
- The user uploads a CSV, or clicks a button to try the built-in messy sample.
- The app runs the full checkup and shows the trust score, the number of problems and the size of the file, followed by the verdict as a large colored card.
- The **Findings** tab lists every problem with its impact on the numbers, why it matters and what to do.
- The **Overview** tab shows the column table and the date coverage.
- The **Clean the file** tab shows the change log, the score after the safe fixes and a download button for the cleaned CSV.

The first part of the code is styling only: the font, the soft blue background, the rounded cards and the pill-shaped tabs. The checking itself comes from the files built above.

In [30]:
%%writefile app.py
import html
import io
import os
from collections import Counter

import pandas as pd
import streamlit as st

import checks
import cleaner
import impact
import scoring
from datagen import make_messy

st.set_page_config(page_title="CheckKro", page_icon="✅", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.stApp, .stApp p, .stApp label, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4,
.stApp button, .stApp input, .stApp textarea, .tagline, .verdict-card, .verdict-card div,
.panel, .panel div, .finding, .finding div, .finding span,
[data-testid="stMetricValue"], [data-testid="stMetricLabel"] {
    font-family: 'Inter', -apple-system, 'Segoe UI', Roboto, sans-serif !important;
}

.stApp p, .stApp li {font-size: 0.95rem; line-height: 1.65; color: #1F2A3C;}
.stApp h1, .stApp h2, .stApp h3 {font-weight: 600; letter-spacing: -0.01em; color: #0B1B3A;}
[data-testid="stMetricLabel"] p {font-size: 0.78rem; font-weight: 500; letter-spacing: 0.06em;
                                 text-transform: uppercase; color: #5B6B85;}
[data-testid="stMetricValue"] {font-size: 1.8rem; color: #0B1B3A; font-weight: 700;}
[data-testid="stExpander"] summary p {font-size: 0.9rem; font-weight: 500; color: #0F7F8B;}

.stApp {background: linear-gradient(160deg, #EAF2FF 0%, #DCE9FB 45%, #E9EEFB 100%);}
header[data-testid="stHeader"] {background: transparent;}
#MainMenu, footer {visibility: hidden;}
.block-container {padding-top: 2rem; max-width: 1150px;}
.tagline {color: #4B5B78; font-size: 1.05rem; margin: 0.3rem 0 1.4rem;}

[data-testid="stMetric"] {
    background: rgba(255,255,255,0.78); border: 1px solid rgba(255,255,255,0.95);
    border-top: 3px solid #0F9AA8; border-radius: 18px; padding: 16px 20px;
    box-shadow: 0 6px 24px rgba(40,80,160,0.08);
}

.verdict-card {display: flex; justify-content: space-between; align-items: center; gap: 24px;
               border-radius: 18px; padding: 16px 24px; margin: 14px 0 18px; color: #FFFFFF;
               box-shadow: 0 8px 24px rgba(40,80,160,0.16);}
.verdict-card .label {font-size: 0.72rem; letter-spacing: 0.1em; text-transform: uppercase; opacity: 0.85;}
.verdict-card .big {font-size: 1.35rem; font-weight: 800;}
.verdict-card .reason {text-align: right; max-width: 55%; font-size: 0.95rem; opacity: 0.97;}
.verdict-card .reason b {font-weight: 800;}

.panel {background: rgba(255,255,255,0.78); border: 1px solid rgba(255,255,255,0.95);
        border-radius: 18px; padding: 16px 20px; box-shadow: 0 6px 24px rgba(40,80,160,0.07);
        margin-bottom: 14px;}
.ptitle {font-weight: 600; color: #0B1B3A; margin-bottom: 8px;}
.brow {display: flex; align-items: center; gap: 10px; margin: 8px 0; font-size: 0.9rem; color: #1F2A3C;}
.blabel {width: 140px; flex: none; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
         font-weight: 700; color: #0B1B3A;}
.btrack {flex: 1; background: #E6EEF9; border-radius: 99px; height: 9px;}
.bfill {height: 9px; border-radius: 99px;}
.bcount {width: 30px; text-align: right; color: #5B6B85;}

.finding {background: rgba(255,255,255,0.82); border: 1px solid rgba(255,255,255,0.95);
          border-radius: 16px; padding: 14px 18px; margin: 12px 0 4px;
          box-shadow: 0 4px 18px rgba(40,80,160,0.07);}
.fhead {display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin-bottom: 6px;}
.pill {font-size: 0.7rem; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase;
       padding: 3px 10px; border-radius: 99px;}
.ftitle {font-weight: 600; color: #0B1B3A;}
.fcol {color: #0B1B3A; font-weight: 700; font-family: ui-monospace, Menlo, monospace; font-size: 0.85rem;
        background: #DDF1F3; padding: 2px 9px; border-radius: 7px;}
.faff {margin-left: auto; color: #5B6B85; font-size: 0.88rem;}
.frow {font-size: 0.92rem; color: #1F2A3C; margin: 3px 0;}
.frow b {color: #0B1B3A; font-weight: 700;}

.tbl {overflow-x: auto; background: rgba(255,255,255,0.78); border: 1px solid rgba(255,255,255,0.95);
        border-radius: 16px; padding: 6px 12px; margin-bottom: 14px;}
.tbl table {border-collapse: collapse; width: 100%; font-size: 0.88rem;}
.tbl th {text-align: left; color: #5B6B85; font-size: 0.72rem; text-transform: uppercase;
         letter-spacing: 0.06em; padding: 9px 10px; border-bottom: 1px solid #D5E1F5; white-space: nowrap;}
.tbl td {padding: 8px 10px; border-bottom: 1px solid #EEF2F8; color: #1F2A3C;}
.tbl td.cn {font-weight: 700; color: #0B1B3A;}

div[data-baseweb="tab-list"] {gap: 8px; background: transparent;}
button[data-baseweb="tab"] {background: rgba(255,255,255,0.65); border: 1px solid #D5E1F5;
                            border-radius: 999px; padding: 6px 18px; height: auto;}
button[data-baseweb="tab"][aria-selected="true"] {background: #0F9AA8; border-color: #0F9AA8;}
button[data-baseweb="tab"][aria-selected="true"] p {color: #FFFFFF;}
div[data-baseweb="tab-highlight"], div[data-baseweb="tab-border"] {display: none;}

[data-testid="stExpander"] {background: rgba(255,255,255,0.6); border: 1px solid rgba(255,255,255,0.9);
                            border-radius: 14px; overflow: hidden;}
[data-testid="stExpander"] details {border: none;}

.stButton > button, .stDownloadButton > button {background: #0F9AA8; color: #FFFFFF; border: none;
                                                border-radius: 12px; padding: 0.55rem 1.3rem;}
.stButton > button p, .stDownloadButton > button p {color: #FFFFFF;}
.stButton > button:hover, .stDownloadButton > button:hover {background: #0B7F8B;}

[data-testid="stFileUploader"] section {background: rgba(255,255,255,0.7); border: 1.5px dashed #8ACBD2;
                                        border-radius: 18px;}
[data-testid="stDataFrame"] {border-radius: 16px; overflow: hidden;}
[data-testid="stAlert"] {border-radius: 14px;}

/* ---------- mobile view ---------- */
@media (max-width: 640px) {
    .block-container {padding: 1rem 0.9rem 3rem;}
    .tagline {font-size: 0.95rem; margin-bottom: 1rem;}

    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) {flex-wrap: wrap; gap: 0.6rem;}
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="stColumn"],
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="column"] {
        flex: 1 1 calc(50% - 0.6rem) !important;
        min-width: calc(50% - 0.6rem) !important;
        width: calc(50% - 0.6rem) !important;
    }
    [data-testid="stMetric"] {padding: 12px 14px;}
    [data-testid="stMetricValue"] {font-size: 1.3rem;}

    .verdict-card {flex-direction: column; align-items: flex-start; gap: 8px; padding: 14px 18px;}
    .verdict-card .reason {text-align: left; max-width: 100%;}
    .verdict-card .big {font-size: 1.15rem;}

    .panel {padding: 14px 16px;}
    .blabel {width: 96px;}
    .finding {padding: 12px 14px;}
    .faff {margin-left: 0; width: 100%;}
    button[data-baseweb="tab"] {padding: 5px 12px;}
    .stButton > button, .stDownloadButton > button {width: 100%;}
}

</style>
""", unsafe_allow_html=True)

TITLES = {
    "missing_values": "Missing values",
    "structural_missing": "Blank by design",
    "disguised_missing": "Placeholder values (like N/A)",
    "exact_duplicates": "Duplicate rows",
    "duplicate_ids": "Repeated IDs",
    "extra_whitespace": "Extra spaces",
    "inconsistent_case": "Inconsistent spelling or case",
    "numeric_as_text": "Numbers stored as text",
    "mixed_types": "Text mixed into numbers",
    "impossible_values": "Impossible values",
    "extreme_outliers": "Extreme outliers",
    "future_dates": "Dates in the future",
    "date_order": "Dates in the wrong order",
    "suspicious_rate": "Suspiciously high rate",
    "constant_column": "Column never changes",
    "all_unique_column": "Every value is unique",
    "messy_column_name": "Messy column name",
    "date_gaps": "Missing days in the data",
    "rare_categories": "Rare values (possible typos)",
}
SEV = {"high": ("#B42318", "#FDECEA", "High"),
       "medium": ("#B45309", "#FEF3E2", "Medium"),
       "low": ("#4B5B78", "#EAF0F8", "Low")}
VERDICT_STYLE = {
    "SAFE TO ANALYZE": "linear-gradient(135deg, #27B5A5, #0F8F97)",
    "USE WITH CAUTION": "linear-gradient(135deg, #F5B54A, #E08A1E)",
    "DO NOT USE UNTIL FIXED": "linear-gradient(135deg, #F0716B, #D6453F)",
}


def esc(x):
    return html.escape(str(x))


def bold_cols(text, columns):
    """Escape text and make any 'column name' in quotes bold."""
    out = html.escape(str(text))
    for c in sorted(map(str, columns), key=len, reverse=True):
        out = out.replace(html.escape(f"'{c}'"), f"<b>{html.escape(c)}</b>")
    return out


def table_html(frame):
    """A simple styled table where the 'column' column is bold."""
    head = "".join(f"<th>{esc(c)}</th>" for c in frame.columns)
    body = ""
    for _, row in frame.iterrows():
        cells = "".join(
            f'<td class="cn">{esc(v)}</td>' if c == "column" else f"<td>{esc(v)}</td>"
            for c, v in zip(frame.columns, row.values))
        body += f"<tr>{cells}</tr>"
    return f'<div class="tbl"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def title_of(check):
    return TITLES.get(check, check.replace("_", " ").capitalize())


def affected(f):
    return f"{f.count:,} rows ({f.pct}%)" if f.count else "Column-level issue"


def bars(rows, colors):
    """rows: list of (label, value, color). Draws simple horizontal bars."""
    top = max([v for _, v, _ in rows] + [1])
    out = ""
    for label, value, color in rows:
        out += (f'<div class="brow"><div class="blabel">{esc(label)}</div>'
                f'<div class="btrack"><div class="bfill" style="width:{value / top * 100:.0f}%;background:{color};"></div></div>'
                f'<div class="bcount">{value}</div></div>')
    return out


def load_csv(file):
    raw = file.getvalue()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError("unreadable")


@st.cache_data(show_spinner="Checking your file...")
def analyse(df):
    findings = checks.run_checkup(df)
    cleaned, log = cleaner.clean(df)
    after = checks.run_checkup(cleaned)
    return findings, cleaned, log, after


# ---------- header and upload ----------
if os.path.exists("logo_clean.png"):
    st.image("logo_clean.png", width=230)
else:
    st.title("CheckKro")
st.markdown('<div class="tagline">Know if your data is safe to analyze.</div>', unsafe_allow_html=True)

uploaded = st.file_uploader("Upload a CSV file", type="csv")
if st.button("Try with a sample messy file"):
    st.session_state["sample"] = True
if uploaded is not None:
    st.session_state["sample"] = False

df = None
if uploaded is not None:
    try:
        df, enc = load_csv(uploaded)
    except Exception:
        st.error("This file could not be read as a CSV. Check that it is a plain CSV export and try again.")
        st.stop()
    if enc != "utf-8-sig":
        st.info(f"This file is not UTF-8, so it was read as {enc}. Special characters may look different.")
elif st.session_state.get("sample"):
    df = make_messy()
    st.info("Showing the built-in sample file.")

if df is None:
    st.stop()
if df.shape[0] == 0 or df.shape[1] == 0:
    st.warning("This file has no rows to check. Upload a file that contains data.")
    st.stop()

findings, cleaned, log, after = analyse(df)
score = scoring.trust_score(findings)
verdict = scoring.verdict(score, findings)
n_sev = {s: sum(f.severity == s for f in findings) for s in ("high", "medium", "low")}
n_high = n_sev["high"]
affected_cols = Counter(f.column for f in findings if f.column != "(all columns)")

# ---------- dashboard: metrics ----------
c1, c2, c3, c4 = st.columns(4)
c1.metric("Trust score", f"{score} / 100")
c2.metric("Problems found", len(findings))
c3.metric("Columns affected", len(affected_cols))
c4.metric("Rows x columns", f"{len(df):,} x {df.shape[1]}")

# ---------- verdict card with reason ----------
if not findings:
    reason = "No problems found."
else:
    lead = ("No serious issues." if n_high == 0 else
            "1 serious issue requires review." if n_high == 1 else
            f"{n_high} serious issues require review.")
    top = findings[0]
    reason = f"{esc(lead)} Biggest: {esc(title_of(top.check))} in <b>{esc(top.column)}</b>."
st.markdown(
    f'<div class="verdict-card" style="background:{VERDICT_STYLE[verdict]};">'
    f'<div><div class="label">Verdict</div><div class="big">{esc(verdict)}</div></div>'
    f'<div class="reason">{reason}</div></div>',
    unsafe_allow_html=True,
)

# ---------- dashboard: breakdown panels ----------
left, right = st.columns(2)
with left:
    rows = [("High", n_sev["high"], "#D6453F"), ("Medium", n_sev["medium"], "#E08A1E"),
            ("Low", n_sev["low"], "#8CA0BF")]
    st.markdown(f'<div class="panel"><div class="ptitle">Findings by severity</div>{bars(rows, None)}</div>',
                unsafe_allow_html=True)
with right:
    top_cols = [(c, n, "#0F9AA8") for c, n in affected_cols.most_common(5)]
    body = bars(top_cols, None) if top_cols else '<div class="frow">No column has problems.</div>'
    st.markdown(f'<div class="panel"><div class="ptitle">Columns needing attention</div>{body}</div>',
                unsafe_allow_html=True)

tab_findings, tab_score, tab_overview, tab_clean = st.tabs(
    ["Findings", "Score explained", "Overview", "Clean the file"])

# ---------- findings ----------
with tab_findings:
    show = st.radio("Show", ["All", "High", "Medium", "Low"], horizontal=True, label_visibility="collapsed")
    shown = [f for f in findings if show == "All" or f.severity == show.lower()]
    if not findings:
        st.success("No problems found.")
    elif not shown:
        st.info("No findings at this severity.")
    for f in shown:
        fg, bg, label = SEV[f.severity]
        st.markdown(
            f'<div class="finding"><div class="fhead">'
            f'<span class="pill" style="color:{fg};background:{bg};">{label}</span>'
            f'<span class="ftitle">{esc(title_of(f.check))}</span>'
            f'<span class="fcol">{esc(f.column)}</span>'
            f'<span class="faff">{esc(affected(f))}</span></div>'
            f'<div class="frow"><b>Why it matters:</b> {bold_cols(f.why, df.columns)}</div>'
            f'<div class="frow"><b>Suggested action:</b> {bold_cols(f.suggestion, df.columns)}</div></div>',
            unsafe_allow_html=True,
        )
        with st.expander("Details and impact on your numbers"):
            st.markdown(f'<div class="frow">{bold_cols(f.detail, df.columns)}</div>', unsafe_allow_html=True)
            effect = impact.impact(df, f)
            if effect:
                st.markdown(f'<div class="frow"><b>Impact on your numbers:</b> {bold_cols(effect, df.columns)}</div>',
                            unsafe_allow_html=True)

# ---------- score explained ----------
with tab_score:
    pen = scoring.PENALTY
    st.markdown("**How the score is calculated.** The score starts at 100. Every finding takes points off, "
                "depending on its severity:")
    for s in ("high", "medium", "low"):
        st.markdown(f"- {n_sev[s]} {s} finding(s) x {pen[s]} points = **-{n_sev[s] * pen[s]}**")
    st.markdown(f"Result: **{score} / 100**.")
    costly = sorted(findings, key=lambda f: -pen[f.severity])[:5]
    if costly:
        st.markdown("**What lowers it most:**")
        for f in costly:
            st.markdown(f"- -{pen[f.severity]}: {title_of(f.check)} in **{f.column}**")
    st.markdown("**What the verdict means:**")
    st.markdown("- **Safe to analyze:** score of 85 or more, and no high-severity findings.\n"
                "- **Use with caution:** score of 60 or more.\n"
                "- **Do not use until fixed:** anything below 60.")
    st.caption("The score is a simple rule-based heuristic: fixed penalties by severity. It is not a statistical "
               "probability and does not guarantee that the data is right or wrong. One high-severity finding "
               "always blocks 'safe to analyze'.")

# ---------- overview ----------
with tab_overview:
    ov = checks.overview(df)
    ov["example"] = ov["example"].astype(str)
    st.subheader("Columns")
    st.markdown(table_html(ov), unsafe_allow_html=True)
    cov = checks.date_coverage(df)
    if len(cov) > 0:
        st.subheader("Date coverage")
        st.markdown(table_html(cov), unsafe_allow_html=True)

# ---------- clean ----------
with tab_clean:
    st.write("Only safe fixes are applied: trimming spaces, turning placeholders like 'N/A' "
             "into real blanks, and removing exact duplicate rows. Everything else needs your decision. "
             "Your original file is never changed.")
    placeholders = int(log.loc[log["action"].str.startswith("placeholder"), "rows_affected"].sum()) if len(log) else 0
    after_score = scoring.trust_score(after)
    m1, m2, m3 = st.columns(3)
    m1.metric("Rows", f"{len(df):,} -> {len(cleaned):,}")
    m2.metric("Placeholders turned blank", f"{placeholders:,}")
    m3.metric("Score after safe fixes", f"{after_score} / 100", delta=after_score - score)
    if len(log) == 0:
        st.info("Nothing needed the safe fixes.")
    else:
        st.subheader("Change log")
        st.markdown(table_html(log), unsafe_allow_html=True)
    st.subheader("Preview of the cleaned data")
    st.dataframe(cleaned.head(50), use_container_width=True)
    d1, d2 = st.columns(2)
    d1.download_button("Download cleaned CSV", cleaned.to_csv(index=False).encode("utf-8"),
                       file_name="cleaned.csv", mime="text/csv")
    d2.download_button("Download change log (CSV)", log.to_csv(index=False).encode("utf-8"),
                       file_name="change_log.csv", mime="text/csv")

st.caption("CheckKro uses rule-based checks. Nothing in your file is changed without a logged record.")

Overwriting app.py


## 14. Testing: how well does it actually work?

This cell creates `evaluate.py`, the test that measures the tool instead of just claiming it works.

It takes a clean dataset, plants **one known error at a time**, and checks whether CheckKro finds that exact error in the right column. There are 17 kinds of planted error, including missing values, hidden "N/A", duplicate rows and IDs, extra spaces, mixed casing, text hiding in a number column, negative amounts, impossible ages, extreme outliers, future dates, delivery before order, a suspicious refund rate, missing days, a constant column and typos in a category.

Each error is planted in three different datasets (3,000, 5,000 and 8,000 rows), which gives 51 tests in total. A test only counts as caught when the right check fires on the right column and finds at least 80% of the planted rows.

The test also runs the checkup on the clean datasets, to count how many false alarms (medium or high findings on healthy data) the tool raises.

In [31]:
%%writefile evaluate.py
import numpy as np
import pandas as pd
import checks
from datagen import make_clean

KINDS = [
    "missing_values", "disguised_missing", "exact_duplicates", "duplicate_ids",
    "extra_whitespace", "inconsistent_case", "mixed_types", "numeric_as_text",
    "negative_amount", "impossible_age", "extreme_outliers", "future_dates",
    "date_order", "suspicious_rate", "date_gaps", "constant_column", "rare_categories",
]

DATASETS = [(3000, 1), (5000, 2), (8000, 3)]   # (rows, seed)


def inject(base, kind, seed):
    """Plant ONE known error. Returns (data, expected_check, expected_column, expected_count)."""
    r = np.random.default_rng(seed)
    df = base.copy()
    n = len(df)

    def pick(frac):
        return r.choice(n, max(int(n * frac), 3), replace=False)

    if kind == "missing_values":
        idx = pick(0.08)
        df["rating"] = df["rating"].astype(float)
        df.loc[idx, "rating"] = np.nan
        return df, "missing_values", "rating", len(idx)

    if kind == "disguised_missing":
        idx = pick(0.05)
        df["city"] = df["city"].astype(object)
        df.loc[idx, "city"] = "N/A"
        return df, "disguised_missing", "city", len(idx)

    if kind == "exact_duplicates":
        copies = df.sample(frac=0.03, random_state=seed)
        df = pd.concat([df, copies], ignore_index=True)
        return df, "exact_duplicates", None, len(copies)

    if kind == "duplicate_ids":
        idx = r.choice(n, 40, replace=False)
        df.loc[idx[:20], "order_id"] = df.loc[idx[20:], "order_id"].values
        return df, "duplicate_ids", "order_id", 20

    if kind == "extra_whitespace":
        idx = pick(0.05)
        df["city"] = df["city"].astype(object)
        df.loc[idx, "city"] = df.loc[idx, "city"].astype(str) + " "
        return df, "extra_whitespace", "city", len(idx)

    if kind == "inconsistent_case":
        idx = pick(0.05)
        df["platform"] = df["platform"].astype(object)
        df.loc[idx, "platform"] = df.loc[idx, "platform"].astype(str).str.lower()
        return df, "inconsistent_case", "platform", len(idx)

    if kind == "mixed_types":
        idx = pick(0.02)
        df["order_amount"] = df["order_amount"].astype(object)
        df.loc[idx, "order_amount"] = "abc"
        return df, "mixed_types", "order_amount", len(idx)

    if kind == "numeric_as_text":
        df["order_amount"] = df["order_amount"].astype(str)
        return df, "numeric_as_text", "order_amount", None

    if kind == "negative_amount":
        idx = pick(0.01)
        df.loc[idx, "order_amount"] = -df.loc[idx, "order_amount"].abs()
        return df, "impossible_values", "order_amount", len(idx)

    if kind == "impossible_age":
        idx = pick(0.01)
        df.loc[idx, "customer_age"] = 150
        return df, "impossible_values", "customer_age", len(idx)

    if kind == "extreme_outliers":
        idx = pick(0.005)
        df.loc[idx, "order_amount"] = df.loc[idx, "order_amount"] * 1000
        return df, "extreme_outliers", "order_amount", len(idx)

    if kind == "future_dates":
        idx = pick(0.01)
        df.loc[idx, "order_date"] = df.loc[idx, "order_date"] + pd.Timedelta(days=1800)
        return df, "future_dates", "order_date", len(idx)

    if kind == "date_order":
        idx = pick(0.02)
        df.loc[idx, "delivery_date"] = df.loc[idx, "order_date"] - pd.Timedelta(days=3)
        return df, "date_order", "delivery_date", len(idx)

    if kind == "suspicious_rate":
        idx = pick(0.40)
        df.loc[idx, "refund_flag"] = 1
        return df, "suspicious_rate", "refund_flag", None

    if kind == "date_gaps":
        start = df["order_date"].min().normalize()
        window = (df["order_date"] >= start + pd.Timedelta(days=100)) & \
                 (df["order_date"] <= start + pd.Timedelta(days=120))
        df = df[~window].reset_index(drop=True)
        return df, "date_gaps", "order_date", 21

    if kind == "constant_column":
        df["platform"] = "Blinkit"
        return df, "constant_column", "platform", None

    if kind == "rare_categories":
        idx = r.choice(n, 3, replace=False)
        df["city"] = df["city"].astype(object)
        df.loc[idx[:2], "city"] = "Dehli"
        df.loc[idx[2], "city"] = "Mumbia"
        return df, "rare_categories", "city", None

    raise ValueError(kind)


def detected(findings, check, column, expected_count):
    """Caught = right check, right column, and it found most of the planted rows."""
    for f in findings:
        if f.check == check and (column is None or f.column == column):
            if expected_count is None or f.count >= 0.8 * expected_count:
                return True
    return False


def run_evaluation():
    rows, false_alarms = [], []
    for size, seed in DATASETS:
        base = make_clean(size, seed)
        serious = [f for f in checks.run_checkup(base) if f.severity in ("medium", "high")]
        false_alarms.append(len(serious))
        for kind in KINDS:
            bad, check, col, exp = inject(base, kind, seed)
            found = checks.run_checkup(bad)
            rows.append({
                "dataset": f"{size} rows",
                "error": kind,
                "caught": detected(found, check, col, exp),
            })
    return pd.DataFrame(rows), false_alarms

Overwriting evaluate.py


**Run the evaluation.** CheckKro caught all 51 of 51 planted errors, across all 17 error types and all 3 dataset sizes, with zero serious false alarms on the clean data. The table below shows True for every error on every dataset.

One honest limit: we wrote both the errors and the checks, on synthetic data, so a perfect score is expected. It shows that each check does what it was built to do. The real test was uploading a real file, the Ola bookings data, which exposed three false-alarm patterns that these tests could not. Those were fixed, and this evaluation was re-run afterwards to confirm that nothing broke.

In [32]:
import evaluate
import importlib
importlib.reload(evaluate)

results, false_alarms = evaluate.run_evaluation()

total = len(results)
caught = int(results["caught"].sum())
print(f"Caught {caught} of {total} planted errors")
print("Serious false alarms on clean data (one number per dataset):", false_alarms)
print()
print(results.pivot(index="error", columns="dataset", values="caught"))

Caught 51 of 51 planted errors
Serious false alarms on clean data (one number per dataset): [0, 0, 0]

dataset            3000 rows  5000 rows  8000 rows
error                                             
constant_column         True       True       True
date_gaps               True       True       True
date_order              True       True       True
disguised_missing       True       True       True
duplicate_ids           True       True       True
exact_duplicates        True       True       True
extra_whitespace        True       True       True
extreme_outliers        True       True       True
future_dates            True       True       True
impossible_age          True       True       True
inconsistent_case       True       True       True
missing_values          True       True       True
mixed_types             True       True       True
negative_amount         True       True       True
numeric_as_text         True       True       True
rare_categories         True  

**Set the app theme.** This cell writes a small settings file, `.streamlit/config.toml`, which Streamlit reads when the app starts. It sets the light base, the blue accent color, the white background and the font style. The last line prints the file back so you can see what was saved.

In [33]:
import os

os.makedirs(".streamlit", exist_ok=True)
theme = """[theme]
base = "light"
primaryColor = "#1A56DB"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F3F6FB"
textColor = "#0B0F19"
font = "sans serif"
"""
with open(".streamlit/config.toml", "w") as f:
    f.write(theme)

print(open(".streamlit/config.toml").read())

[theme]
base = "light"
primaryColor = "#1A56DB"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F3F6FB"
textColor = "#0B0F19"
font = "sans serif"



 **Prepare the logo.** This cell takes the uploaded `logo.png` and removes its white background, so the logo sits cleanly on the soft blue page. It works out how far each pixel is from white and makes the white parts transparent, keeping the soft edges of the letters. It then trims the empty space around the logo, resizes it to a width of 600 pixels and saves it as `logo_clean.png`, which the app picks up automatically. The original `logo.png` is left untouched.

In [34]:
from PIL import Image
import numpy as np

im = Image.open("logo.png").convert("RGBA")
a = np.array(im).astype(float)

dist = 255 - a[..., :3].min(axis=2)      # how far each pixel is from white
alpha = np.clip(dist * 1.6, 0, 255)      # the further from white, the more visible
alpha[dist < 6] = 0                      # pure white becomes fully transparent
a[..., 3] = alpha

res = Image.fromarray(a.astype("uint8"), "RGBA")
res = res.crop(res.getbbox())            # trim the empty space around the logo
res = res.resize((600, int(res.height * 600 / res.width)), Image.LANCZOS)
res.save("logo_clean.png")
print("Saved logo_clean.png", res.size)

Saved logo_clean.png (600, 235)


**List the libraries.** This writes `requirements.txt`, which tells the hosting service which packages the app needs: Streamlit for the web page, and pandas and numpy for the data work.

In [35]:
%%writefile requirements.txt
streamlit>=1.40
pandas>=2.0
numpy>=1.24

Overwriting requirements.txt


**Write the README.** This saves the project description, results and limits as `README.md`, the first page anyone sees on GitHub.

In [36]:
%%writefile README.md
# CheckKro

**Know if your data is safe to analyze.**

CheckKro checks any CSV file the way an analyst would on first opening it, and answers one question: can I trust this data? It gives a trust score out of 100, a verdict, and what each problem does to your numbers.

Live demo: (add the link after deploying)

## What it does

1. **Checks** the file for the problems analysts look for first: missing and hidden-missing values, duplicate rows and IDs, formatting mistakes, impossible values, extreme outliers, date errors and suspicious patterns.
2. **Scores and judges.** Every file gets a trust score and one of three verdicts: safe to analyze, use with caution, or do not use until fixed.
3. **Explains the impact on your numbers.** Not just "150 duplicate rows", but "total order amount is overstated by 3.1%".
4. **Cleans safely.** It only trims spaces, turns placeholders like "N/A" and "#NAME?" into real blanks, and removes exact duplicates. Every change is logged. Risky fixes are left to a human.

## What makes it different

- **Impact, not just detection.** Each finding says what it does to your results.
- **It understands blanks that are by design.** A cancelled ride has no rating, and that is not an error. CheckKro spots blanks that follow a rule in another column and rates them low instead of raising an alarm.
- **Suspicious-data detection.** Flags rates that look too high to be real, such as a 42% refund rate.
- **No silent changes.** The cleaner never guesses, and every change appears in a log.
- **Rule-based, not AI.** The checks are plain Python, so every result is repeatable and explainable.

## How well does it work?

**Planted-error test.** I planted 17 kinds of known error, one at a time, in 3 datasets of different sizes (51 tests in total). CheckKro caught 51 of 51, with 0 serious false alarms on the clean datasets. These errors were written by me on synthetic data, so this shows each check does what it was built to do, not how it performs on every real file.

**Real-file test.** I then ran it on a real dataset of 103,024 ride bookings. The first version scored it 0 out of 100 with 9 high-severity alarms, and most of them were wrong:
- Columns like ratings and payment method are blank for cancelled rides. That is by design.
- Customer ID repeats because customers book more than once.
- A column full of Excel `#NAME?` errors was under-reported.

I fixed all three, and re-ran the planted-error test to confirm nothing broke. The same file now scores 72 out of 100 with one high-severity finding: the `#NAME?` column, which is a real problem.

## Known limits

- Outliers are found with a simple statistical rule, so naturally skewed data (such as booking values with a long tail) can be flagged even when the values are real.
- Some checks use column names to decide what a column means (for example, "amount" or "age"), so they work best on English column names.
- The thresholds, such as a refund rate above 25%, are sensible defaults, not universal truths.
- It reads CSV files only.

## Run it yourself

    pip install -r requirements.txt
    streamlit run app.py

Click "Try with a sample messy file" to see it work without uploading anything.

## Files

- `checks.py` holds all the checks and the `run_checkup` function.
- `scoring.py` turns findings into a trust score and a verdict.
- `cleaner.py` applies the safe fixes and writes the change log.
- `impact.py` writes the plain-language impact of each finding.
- `app.py` is the Streamlit web app.
- `datagen.py` builds the clean and messy test data.
- `evaluate.py` plants errors and measures how many are caught.
- `step1.ipynb` is the notebook the project was built in, step by step.

## Author

Disha Jaiswal, data analyst.

Overwriting README.md


**Build the chart tool.** This creates `charts.py`, which draws quick charts from the uploaded file. Settings sit on the left and the chart on the right: chart type (bar, line, scatter or histogram), the column to group by, what to measure and how to combine it, how many top groups to show, the bar order, value labels and an editable title. You can chart the original or the cleaned data, long date ranges are grouped by day, week or month, and the plotted numbers can be downloaded as a CSV.

In [37]:
%%writefile charts.py
import altair as alt
import numpy as np
import pandas as pd

import checks

alt.data_transformers.disable_max_rows()

TEAL = "#0F9AA8"
COUNT = "(count of rows)"
PERIOD = {"Day": "D", "Week": "W", "Month": "M"}


def is_num(s):
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)


def style(chart, title):
    return (chart.properties(title=title, height=340, background="transparent")
            .configure_axis(labelColor="#4B5B78", titleColor="#0B1B3A", gridColor="#E3ECF8",
                            domainColor="#C9D8EE", labelFontSize=12, titleFontSize=12)
            .configure_title(color="#0B1B3A", fontSize=15, anchor="start")
            .configure_view(stroke=None))


def auto_title(kind, x, y=None, agg="sum"):
    if kind in ("Bar", "Line"):
        ylabel = "Number of rows" if y == COUNT else f"{agg} of {y}"
        return f"{ylabel} by {x}"
    if kind == "Scatter":
        return f"{y} vs {x}"
    return f"Distribution of {x}"


def build_chart(df, kind, x, y=None, agg="sum", top_n=10, bins=30, freq="Day", dates=None,
                title=None, sort_by="Largest first", labels=True):
    """Returns (chart, plotted_data, note). chart is None when nothing can be drawn."""
    dates = dates or {}

    if kind in ("Bar", "Line"):
        is_date = x in dates
        if is_date:
            label = dates[x].dt.to_period(PERIOD[freq]).dt.to_timestamp()
        else:
            label = df[x] if is_num(df[x]) else df[x].astype(str)
        if y == COUNT:
            g = pd.DataFrame({"label": label}).dropna().groupby("label").size().reset_index(name="value")
            ylabel = "Number of rows"
        else:
            g = (pd.DataFrame({"label": label, "v": df[y]}).dropna()
                 .groupby("label")["v"].agg(agg).reset_index(name="value"))
            ylabel = f"{agg} of {y}"
        if g.empty:
            return None, None, "There is nothing to plot for this selection."
        if kind == "Bar" and not is_date:
            g = g.sort_values("value", ascending=False).head(top_n)
            g["label"] = g["label"].astype(str)
            if sort_by == "Smallest first":
                g = g.sort_values("value")
            elif sort_by == "A to Z":
                g = g.sort_values("label")
        else:
            g = g.sort_values("label")
        if is_date:
            xenc = alt.X("label:T", title=x)
        elif kind == "Line" and is_num(g["label"]):
            xenc = alt.X("label:Q", title=x)
        else:
            xenc = alt.X("label:N", sort=None, title=x)
        base = alt.Chart(g)
        if kind == "Bar":
            mark = base.mark_bar(color=TEAL, cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
        else:
            mark = base.mark_line(color=TEAL, strokeWidth=2.5, point=alt.OverlayMarkDef(color=TEAL))
        chart = mark.encode(x=xenc, y=alt.Y("value:Q", title=ylabel),
                            tooltip=[alt.Tooltip("label", title=x),
                                     alt.Tooltip("value:Q", title=ylabel, format=",.2f")])
        if labels and len(g) <= 25:
            whole = bool((g["value"] % 1 == 0).all())
            text = base.mark_text(dy=-9, fontSize=11, fontWeight="bold", color="#0B1B3A").encode(
                x=xenc, y=alt.Y("value:Q", title=ylabel),
                text=alt.Text("value:Q", format=",.0f" if whole else ",.1f"))
            chart = chart + text
        note = f"Showing the top {top_n} values." if kind == "Bar" and not is_date and len(g) == top_n else None
        return (style(chart, title or auto_title(kind, x, y, agg)),
                g.rename(columns={"label": x, "value": ylabel}), note)

    if kind == "Scatter":
        if x == y:
            return None, None, "Pick two different columns."
        d = df[[x, y]].dropna().rename(columns={x: "x", y: "y"})
        if d.empty:
            return None, None, "There is nothing to plot for this selection."
        note = None
        if len(d) > 3000:
            note = f"Showing a random sample of 3,000 points out of {len(d):,}."
            d = d.sample(3000, random_state=0)
        chart = (alt.Chart(d).mark_circle(size=45, opacity=0.45, color=TEAL)
                 .encode(x=alt.X("x:Q", title=x), y=alt.Y("y:Q", title=y),
                         tooltip=[alt.Tooltip("x:Q", title=x), alt.Tooltip("y:Q", title=y)]))
        return style(chart, title or auto_title(kind, x, y)), d.rename(columns={"x": x, "y": y}), note

    # Histogram
    s = pd.to_numeric(df[x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return None, None, "There are no numbers to plot in this column."
    counts, edges = np.histogram(s, bins=bins)
    h = pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})
    chart = (alt.Chart(h).mark_bar(color=TEAL, opacity=0.9)
             .encode(x=alt.X("start:Q", title=x), x2="end:Q",
                     y=alt.Y("count:Q", title="Number of rows"),
                     tooltip=[alt.Tooltip("start:Q", title="From", format=",.2f"),
                              alt.Tooltip("end:Q", title="To", format=",.2f"),
                              alt.Tooltip("count:Q", title="Rows")]))
    return style(chart, title or auto_title(kind, x)), h, None


def render_charts(df, cleaned):
    import streamlit as st

    left, right = st.columns([1, 2.2])
    problem = None
    chart = plotted = note = None

    with left:
        st.markdown("**Chart settings**")
        source = st.radio("Data", ["Original", "Cleaned"], horizontal=True, key="ch_source")
        data = df if source == "Original" else cleaned
        numeric = [c for c in data.columns if is_num(data[c])]
        dates = checks.date_columns(data)
        cols = list(data.columns)
        kind = st.selectbox("Chart type", ["Bar", "Line", "Scatter", "Histogram"], key="ch_kind")

        x = y = None
        agg, top_n, bins, freq = "sum", 10, 30, "Day"
        sort_by, labels = "Largest first", True

        if kind in ("Bar", "Line"):
            guess = next((i for i, c in enumerate(cols)
                          if not is_num(data[c]) and 2 <= data[c].nunique() <= 50), 0)
            x = st.selectbox("Group by column", cols, index=guess, key="ch_x")
            y = st.selectbox("Y value", [COUNT] + numeric, key="ch_y")
            if y != COUNT:
                agg = st.selectbox("Combine values by", ["sum", "mean", "median", "max", "min"], key="ch_agg")
            if x in dates:
                span = (dates[x].max() - dates[x].min()).days if dates[x].notna().any() else 0
                default = 2 if span > 180 else 1 if span > 45 else 0
                freq = st.selectbox("Group dates by", ["Day", "Week", "Month"], index=default, key="ch_freq")
            elif kind == "Bar":
                top_n = st.slider("Top groups", 3, 30, 10, key="ch_top")
                sort_by = st.selectbox("Order bars", ["Largest first", "Smallest first", "A to Z"], key="ch_sort")
            labels = st.checkbox("Show value labels", value=True, key="ch_labels")
        elif kind == "Scatter":
            if len(numeric) < 2:
                problem = "A scatter chart needs at least two numeric columns."
            else:
                x = st.selectbox("X axis", numeric, key="ch_sx")
                y = st.selectbox("Y axis", numeric, index=1, key="ch_sy")
        else:
            if not numeric:
                problem = "A histogram needs a numeric column."
            else:
                x = st.selectbox("Column", numeric, key="ch_hx")
                bins = st.slider("Number of bars", 10, 60, 30, key="ch_bins")

        if problem is None:
            title = st.text_input("Chart title", auto_title(kind, x, y, agg),
                                  key=f"ch_title_{kind}_{x}_{y}_{agg}")

    with right:
        if problem is None:
            chart, plotted, note = build_chart(data, kind, x, y, agg, top_n, bins, freq, dates,
                                               title=title, sort_by=sort_by, labels=labels)
            if chart is None:
                problem = note
        if problem is not None:
            st.info(problem)
        else:
            st.altair_chart(chart, use_container_width=True, theme=None)
            if note:
                st.caption(note)
            st.caption(f"This chart uses the {source.lower()} file.")
            st.download_button("Download chart data (CSV)", plotted.to_csv(index=False).encode("utf-8"),
                               file_name="chart_data.csv", mime="text/csv", key="ch_dl")

Overwriting charts.py


**Prepare the dark-mode logo.** The navy letters of the logo would disappear on a dark page, so this cell makes a second copy, `logo_dark.png`, with light letters and the same teal slash. The app shows whichever one suits the device's light or dark setting.

In [38]:
from PIL import Image
import numpy as np

im = Image.open("logo_clean.png").convert("RGBA")
a = np.array(im).astype(float)

# navy letters have a low green value, the teal slash has a high one
t = np.clip((a[..., 1] - 74) / (190 - 74), 0, 1)     # 0 = navy, 1 = teal
light = np.array([232, 238, 248], dtype=float)
for i in range(3):
    a[..., i] = light[i] * (1 - t) + a[..., i] * t   # navy becomes light, teal stays

Image.fromarray(a.astype("uint8"), "RGBA").save("logo_dark.png")
print("Saved logo_dark.png")

Saved logo_dark.png


**Let the theme follow the device.** This rewrites the theme file so it no longer forces a light page. Streamlit now follows the device's light or dark setting, and only the teal accent color is fixed.

In [39]:
import os

os.makedirs(".streamlit", exist_ok=True)
theme = """[theme]
primaryColor = "#0F9AA8"
font = "sans serif"
"""
with open(".streamlit/config.toml", "w") as f:
    f.write(theme)

print(open(".streamlit/config.toml").read())

[theme]
primaryColor = "#0F9AA8"
font = "sans serif"



**Make the charts readable in light and dark.** This updates `charts.py` so the axis labels, titles and bar numbers use a soft grey-blue that reads on both a light and a dark page. The teal bars and lines stay the same.

In [40]:
%%writefile charts.py
import altair as alt
import numpy as np
import pandas as pd

import checks

alt.data_transformers.disable_max_rows()

TEAL = "#0F9AA8"
NEUTRAL = "#7A879E"   # reads on both light and dark backgrounds
COUNT = "(count of rows)"
PERIOD = {"Day": "D", "Week": "W", "Month": "M"}


def is_num(s):
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)


def style(chart, title):
    return (chart.properties(title=title, height=340, background="transparent")
            .configure_axis(labelColor=NEUTRAL, titleColor=NEUTRAL, gridColor="rgba(128,128,128,0.22)",
                            domainColor="rgba(128,128,128,0.4)", labelFontSize=12, titleFontSize=12)
            .configure_title(color=NEUTRAL, fontSize=15, fontWeight=600, anchor="start")
            .configure_view(stroke=None))


def auto_title(kind, x, y=None, agg="sum"):
    if kind in ("Bar", "Line"):
        ylabel = "Number of rows" if y == COUNT else f"{agg} of {y}"
        return f"{ylabel} by {x}"
    if kind == "Scatter":
        return f"{y} vs {x}"
    return f"Distribution of {x}"


def build_chart(df, kind, x, y=None, agg="sum", top_n=10, bins=30, freq="Day", dates=None,
                title=None, sort_by="Largest first", labels=True):
    """Returns (chart, plotted_data, note). chart is None when nothing can be drawn."""
    dates = dates or {}

    if kind in ("Bar", "Line"):
        is_date = x in dates
        if is_date:
            label = dates[x].dt.to_period(PERIOD[freq]).dt.to_timestamp()
        else:
            label = df[x] if is_num(df[x]) else df[x].astype(str)
        if y == COUNT:
            g = pd.DataFrame({"label": label}).dropna().groupby("label").size().reset_index(name="value")
            ylabel = "Number of rows"
        else:
            g = (pd.DataFrame({"label": label, "v": df[y]}).dropna()
                 .groupby("label")["v"].agg(agg).reset_index(name="value"))
            ylabel = f"{agg} of {y}"
        if g.empty:
            return None, None, "There is nothing to plot for this selection."
        if kind == "Bar" and not is_date:
            g = g.sort_values("value", ascending=False).head(top_n)
            g["label"] = g["label"].astype(str)
            if sort_by == "Smallest first":
                g = g.sort_values("value")
            elif sort_by == "A to Z":
                g = g.sort_values("label")
        else:
            g = g.sort_values("label")
        if is_date:
            xenc = alt.X("label:T", title=x)
        elif kind == "Line" and is_num(g["label"]):
            xenc = alt.X("label:Q", title=x)
        else:
            xenc = alt.X("label:N", sort=None, title=x)
        base = alt.Chart(g)
        if kind == "Bar":
            mark = base.mark_bar(color=TEAL, cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
        else:
            mark = base.mark_line(color=TEAL, strokeWidth=2.5, point=alt.OverlayMarkDef(color=TEAL))
        chart = mark.encode(x=xenc, y=alt.Y("value:Q", title=ylabel),
                            tooltip=[alt.Tooltip("label", title=x),
                                     alt.Tooltip("value:Q", title=ylabel, format=",.2f")])
        if labels and len(g) <= 25:
            whole = bool((g["value"] % 1 == 0).all())
            text = base.mark_text(dy=-9, fontSize=11, fontWeight="bold", color=NEUTRAL).encode(
                x=xenc, y=alt.Y("value:Q", title=ylabel),
                text=alt.Text("value:Q", format=",.0f" if whole else ",.1f"))
            chart = chart + text
        note = f"Showing the top {top_n} values." if kind == "Bar" and not is_date and len(g) == top_n else None
        return (style(chart, title or auto_title(kind, x, y, agg)),
                g.rename(columns={"label": x, "value": ylabel}), note)

    if kind == "Scatter":
        if x == y:
            return None, None, "Pick two different columns."
        d = df[[x, y]].dropna().rename(columns={x: "x", y: "y"})
        if d.empty:
            return None, None, "There is nothing to plot for this selection."
        note = None
        if len(d) > 3000:
            note = f"Showing a random sample of 3,000 points out of {len(d):,}."
            d = d.sample(3000, random_state=0)
        chart = (alt.Chart(d).mark_circle(size=45, opacity=0.45, color=TEAL)
                 .encode(x=alt.X("x:Q", title=x), y=alt.Y("y:Q", title=y),
                         tooltip=[alt.Tooltip("x:Q", title=x), alt.Tooltip("y:Q", title=y)]))
        return style(chart, title or auto_title(kind, x, y)), d.rename(columns={"x": x, "y": y}), note

    # Histogram
    s = pd.to_numeric(df[x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return None, None, "There are no numbers to plot in this column."
    counts, edges = np.histogram(s, bins=bins)
    h = pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})
    chart = (alt.Chart(h).mark_bar(color=TEAL, opacity=0.9)
             .encode(x=alt.X("start:Q", title=x), x2="end:Q",
                     y=alt.Y("count:Q", title="Number of rows"),
                     tooltip=[alt.Tooltip("start:Q", title="From", format=",.2f"),
                              alt.Tooltip("end:Q", title="To", format=",.2f"),
                              alt.Tooltip("count:Q", title="Rows")]))
    return style(chart, title or auto_title(kind, x)), h, None


def render_charts(df, cleaned):
    import streamlit as st

    left, right = st.columns([1, 2.2])
    problem = None
    chart = plotted = note = None

    with left:
        st.markdown("**Chart settings**")
        source = st.radio("Data", ["Original", "Cleaned"], horizontal=True, key="ch_source")
        data = df if source == "Original" else cleaned
        numeric = [c for c in data.columns if is_num(data[c])]
        dates = checks.date_columns(data)
        cols = list(data.columns)
        kind = st.selectbox("Chart type", ["Bar", "Line", "Scatter", "Histogram"], key="ch_kind")

        x = y = None
        agg, top_n, bins, freq = "sum", 10, 30, "Day"
        sort_by, labels = "Largest first", True

        if kind in ("Bar", "Line"):
            guess = next((i for i, c in enumerate(cols)
                          if not is_num(data[c]) and 2 <= data[c].nunique() <= 50), 0)
            x = st.selectbox("Group by column", cols, index=guess, key="ch_x")
            y = st.selectbox("Y value", [COUNT] + numeric, key="ch_y")
            if y != COUNT:
                agg = st.selectbox("Combine values by", ["sum", "mean", "median", "max", "min"], key="ch_agg")
            if x in dates:
                span = (dates[x].max() - dates[x].min()).days if dates[x].notna().any() else 0
                default = 2 if span > 180 else 1 if span > 45 else 0
                freq = st.selectbox("Group dates by", ["Day", "Week", "Month"], index=default, key="ch_freq")
            elif kind == "Bar":
                top_n = st.slider("Top groups", 3, 30, 10, key="ch_top")
                sort_by = st.selectbox("Order bars", ["Largest first", "Smallest first", "A to Z"], key="ch_sort")
            labels = st.checkbox("Show value labels", value=True, key="ch_labels")
        elif kind == "Scatter":
            if len(numeric) < 2:
                problem = "A scatter chart needs at least two numeric columns."
            else:
                x = st.selectbox("X axis", numeric, key="ch_sx")
                y = st.selectbox("Y axis", numeric, index=1, key="ch_sy")
        else:
            if not numeric:
                problem = "A histogram needs a numeric column."
            else:
                x = st.selectbox("Column", numeric, key="ch_hx")
                bins = st.slider("Number of bars", 10, 60, 30, key="ch_bins")

        if problem is None:
            title = st.text_input("Chart title", auto_title(kind, x, y, agg),
                                  key=f"ch_title_{kind}_{x}_{y}_{agg}")

    with right:
        if problem is None:
            chart, plotted, note = build_chart(data, kind, x, y, agg, top_n, bins, freq, dates,
                                               title=title, sort_by=sort_by, labels=labels)
            if chart is None:
                problem = note
        if problem is not None:
            st.info(problem)
        else:
            st.altair_chart(chart, use_container_width=True, theme=None)
            if note:
                st.caption(note)
            st.caption(f"This chart uses the {source.lower()} file.")
            st.download_button("Download chart data (CSV)", plotted.to_csv(index=False).encode("utf-8"),
                               file_name="chart_data.csv", mime="text/csv", key="ch_dl")

Overwriting charts.py


**Light and dark mode.** The app now follows the device's light or dark setting automatically, on both phone and laptop. All colors are defined once at the top and switch together: the page, cards, tables, severity tags, buttons and tabs. The logo has a light-lettered copy for dark mode, and the chart text uses a neutral color that reads on both. Warnings stay amber and red in both modes.

In [41]:
%%writefile app.py
import base64
import html
import io
import os
from collections import Counter

import pandas as pd
import streamlit as st

import charts
import checks
import cleaner
import impact
import scoring
from datagen import make_messy

st.set_page_config(page_title="CheckKro", page_icon="✅", layout="wide")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

:root {
    --bg1: #EEF4F8; --bg2: #DDE9F4;
    --card: rgba(255,255,255,0.80); --card-line: rgba(255,255,255,0.95);
    --text: #0B1B3A; --body: #1F2A3C; --muted: #5B6B85;
    --accent: #0F9AA8; --accent-hover: #0B7F8B; --on-accent: #FFFFFF;
    --accent-soft: #DDF1F3; --accent-text: #0F7F8B;
    --track: #E6EEF9; --line: #D5E1F5; --shadow: rgba(40,80,160,0.09);
    --tab: rgba(255,255,255,0.65);
    --hi-fg: #B42318; --hi-bg: #FDECEA; --me-fg: #B45309; --me-bg: #FEF3E2; --lo-fg: #4B5B78; --lo-bg: #EAF0F8;
}
@media (prefers-color-scheme: dark) {
    :root {
        --bg1: #0A1020; --bg2: #0E1830;
        --card: rgba(18,27,46,0.88); --card-line: rgba(255,255,255,0.07);
        --text: #E8EEF8; --body: #CBD5E6; --muted: #8FA0BC;
        --accent: #2DD4BF; --accent-hover: #5EEAD4; --on-accent: #05211D;
        --accent-soft: #12303A; --accent-text: #5EEAD4;
        --track: #1C2740; --line: #27344F; --shadow: rgba(0,0,0,0.4);
        --tab: rgba(18,27,46,0.7);
        --hi-fg: #F0716B; --hi-bg: #3A1A1C; --me-fg: #F5B54A; --me-bg: #3A2A12; --lo-fg: #9FB0CC; --lo-bg: #1C2740;
    }
}

.stApp, .stApp p, .stApp label, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4,
.stApp button, .stApp input, .stApp textarea, .tagline, .verdict-card, .verdict-card div,
.panel, .panel div, .finding, .finding div, .finding span,
[data-testid="stMetricValue"], [data-testid="stMetricLabel"] {
    font-family: 'Inter', -apple-system, 'Segoe UI', Roboto, sans-serif !important;
}

.stApp {background: linear-gradient(160deg, var(--bg1) 0%, var(--bg2) 100%);}
header[data-testid="stHeader"] {background: transparent;}
#MainMenu, footer {visibility: hidden;}
.block-container {padding-top: 2rem; max-width: 1150px;}

.stApp p, .stApp li {font-size: 0.95rem; line-height: 1.65; color: var(--body);}
.stApp h1, .stApp h2, .stApp h3 {font-weight: 600; letter-spacing: -0.01em; color: var(--text);}
.tagline {color: var(--muted); font-size: 1.05rem; margin: 0.3rem 0 1.4rem;}

.logo {width: 230px; display: block;}
.logo-dark {display: none;}
@media (prefers-color-scheme: dark) {
    .logo-light {display: none;}
    .logo-dark {display: block;}
}

[data-testid="stMetricLabel"] p {font-size: 0.78rem; font-weight: 500; letter-spacing: 0.06em;
                                 text-transform: uppercase; color: var(--muted);}
[data-testid="stMetricValue"] {font-size: 1.8rem; color: var(--text); font-weight: 700;}
[data-testid="stMetric"] {
    background: var(--card); border: 1px solid var(--card-line);
    border-top: 3px solid var(--accent); border-radius: 18px; padding: 16px 20px;
    box-shadow: 0 6px 24px var(--shadow);
}

.verdict-card {display: flex; justify-content: space-between; align-items: center; gap: 24px;
               border-radius: 18px; padding: 16px 24px; margin: 14px 0 18px; color: #FFFFFF;
               box-shadow: 0 8px 24px var(--shadow);}
.verdict-card .label {font-size: 0.72rem; letter-spacing: 0.1em; text-transform: uppercase; opacity: 0.85;}
.verdict-card .big {font-size: 1.35rem; font-weight: 800;}
.verdict-card .reason {text-align: right; max-width: 55%; font-size: 0.95rem; opacity: 0.97;}
.verdict-card .reason b {font-weight: 800;}

.panel {background: var(--card); border: 1px solid var(--card-line);
        border-radius: 18px; padding: 16px 20px; box-shadow: 0 6px 24px var(--shadow);
        margin-bottom: 14px;}
.ptitle {font-weight: 600; color: var(--text); margin-bottom: 8px;}
.brow {display: flex; align-items: center; gap: 10px; margin: 8px 0; font-size: 0.9rem; color: var(--body);}
.blabel {width: 140px; flex: none; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
         font-weight: 700; color: var(--text);}
.btrack {flex: 1; background: var(--track); border-radius: 99px; height: 9px;}
.bfill {height: 9px; border-radius: 99px;}
.bcount {width: 30px; text-align: right; color: var(--muted);}

.finding {background: var(--card); border: 1px solid var(--card-line);
          border-radius: 16px; padding: 14px 18px; margin: 12px 0 4px;
          box-shadow: 0 4px 18px var(--shadow);}
.fhead {display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin-bottom: 6px;}
.pill {font-size: 0.7rem; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase;
       padding: 3px 10px; border-radius: 99px;}
.pill-hi {color: var(--hi-fg); background: var(--hi-bg);}
.pill-me {color: var(--me-fg); background: var(--me-bg);}
.pill-lo {color: var(--lo-fg); background: var(--lo-bg);}
.ftitle {font-weight: 600; color: var(--text);}
.fcol {color: var(--text); font-weight: 700; font-family: ui-monospace, Menlo, monospace; font-size: 0.85rem;
       background: var(--accent-soft); padding: 2px 9px; border-radius: 7px;}
.faff {margin-left: auto; color: var(--muted); font-size: 0.88rem;}
.frow {font-size: 0.92rem; color: var(--body); margin: 3px 0;}
.frow b {color: var(--text); font-weight: 700;}

.tbl {overflow-x: auto; background: var(--card); border: 1px solid var(--card-line);
      border-radius: 16px; padding: 6px 12px; margin-bottom: 14px;}
.tbl table {border-collapse: collapse; width: 100%; font-size: 0.88rem;}
.tbl th {text-align: left; color: var(--muted); font-size: 0.72rem; text-transform: uppercase;
         letter-spacing: 0.06em; padding: 9px 10px; border-bottom: 1px solid var(--line); white-space: nowrap;}
.tbl td {padding: 8px 10px; border-bottom: 1px solid var(--line); color: var(--body);}
.tbl td.cn {font-weight: 700; color: var(--text);}

div[data-baseweb="tab-list"] {gap: 8px; background: transparent;}
button[data-baseweb="tab"] {background: var(--tab); border: 1px solid var(--line);
                            border-radius: 999px; padding: 6px 18px; height: auto;}
button[data-baseweb="tab"][aria-selected="true"] {background: var(--accent); border-color: var(--accent);}
button[data-baseweb="tab"][aria-selected="true"] p {color: var(--on-accent);}
div[data-baseweb="tab-highlight"], div[data-baseweb="tab-border"] {display: none;}

[data-testid="stExpander"] {background: var(--card); border: 1px solid var(--card-line);
                            border-radius: 14px; overflow: hidden;}
[data-testid="stExpander"] details {border: none;}
[data-testid="stExpander"] summary p {font-size: 0.9rem; font-weight: 500; color: var(--accent-text);}

.stButton > button, .stDownloadButton > button {background: var(--accent); color: var(--on-accent); border: none;
                                                border-radius: 12px; padding: 0.55rem 1.3rem;}
.stButton > button p, .stDownloadButton > button p {color: var(--on-accent);}
.stButton > button:hover, .stDownloadButton > button:hover {background: var(--accent-hover);}

[data-testid="stFileUploader"] section {background: var(--card); border: 1.5px dashed var(--accent);
                                        border-radius: 18px;}
[data-testid="stDataFrame"] {border-radius: 16px; overflow: hidden;}
[data-testid="stAlert"] {border-radius: 14px;}

/* ---------- mobile view ---------- */
@media (max-width: 640px) {
    .block-container {padding: 1rem 0.9rem 3rem;}
    .tagline {font-size: 0.95rem; margin-bottom: 1rem;}
    .logo {width: 180px;}

    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) {flex-wrap: wrap; gap: 0.6rem;}
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="stColumn"],
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="column"] {
        flex: 1 1 calc(50% - 0.6rem) !important;
        min-width: calc(50% - 0.6rem) !important;
        width: calc(50% - 0.6rem) !important;
    }
    [data-testid="stMetric"] {padding: 12px 14px;}
    [data-testid="stMetricValue"] {font-size: 1.3rem;}

    .verdict-card {flex-direction: column; align-items: flex-start; gap: 8px; padding: 14px 18px;}
    .verdict-card .reason {text-align: left; max-width: 100%;}
    .verdict-card .big {font-size: 1.15rem;}

    .panel {padding: 14px 16px;}
    .blabel {width: 96px;}
    .finding {padding: 12px 14px;}
    .faff {margin-left: 0; width: 100%;}
    button[data-baseweb="tab"] {padding: 5px 12px;}
    .stButton > button, .stDownloadButton > button {width: 100%;}
}
</style>
""", unsafe_allow_html=True)

TITLES = {
    "missing_values": "Missing values",
    "structural_missing": "Blank by design",
    "disguised_missing": "Placeholder values (like N/A)",
    "exact_duplicates": "Duplicate rows",
    "duplicate_ids": "Repeated IDs",
    "extra_whitespace": "Extra spaces",
    "inconsistent_case": "Inconsistent spelling or case",
    "numeric_as_text": "Numbers stored as text",
    "mixed_types": "Text mixed into numbers",
    "impossible_values": "Impossible values",
    "extreme_outliers": "Extreme outliers",
    "future_dates": "Dates in the future",
    "date_order": "Dates in the wrong order",
    "suspicious_rate": "Suspiciously high rate",
    "constant_column": "Column never changes",
    "all_unique_column": "Every value is unique",
    "messy_column_name": "Messy column name",
    "date_gaps": "Missing days in the data",
    "rare_categories": "Rare values (possible typos)",
}
SEV = {"high": ("hi", "High"), "medium": ("me", "Medium"), "low": ("lo", "Low")}
VERDICT_STYLE = {
    "SAFE TO ANALYZE": "linear-gradient(135deg, #27B5A5, #0F8F97)",
    "USE WITH CAUTION": "linear-gradient(135deg, #F5B54A, #E08A1E)",
    "DO NOT USE UNTIL FIXED": "linear-gradient(135deg, #F0716B, #D6453F)",
}


def esc(x):
    return html.escape(str(x))


def bold_cols(text, columns):
    """Escape text and make any 'column name' in quotes bold."""
    out = html.escape(str(text))
    for c in sorted(map(str, columns), key=len, reverse=True):
        out = out.replace(html.escape(f"'{c}'"), f"<b>{html.escape(c)}</b>")
    return out


def table_html(frame):
    """A simple styled table where the 'column' column is bold."""
    head = "".join(f"<th>{esc(c)}</th>" for c in frame.columns)
    body = ""
    for _, row in frame.iterrows():
        cells = "".join(
            f'<td class="cn">{esc(v)}</td>' if c == "column" else f"<td>{esc(v)}</td>"
            for c, v in zip(frame.columns, row.values))
        body += f"<tr>{cells}</tr>"
    return f'<div class="tbl"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def title_of(check):
    return TITLES.get(check, check.replace("_", " ").capitalize())


def affected(f):
    return f"{f.count:,} rows ({f.pct}%)" if f.count else "Column-level issue"


def bars(rows, colors):
    """rows: list of (label, value, color). Draws simple horizontal bars."""
    top = max([v for _, v, _ in rows] + [1])
    out = ""
    for label, value, color in rows:
        out += (f'<div class="brow"><div class="blabel">{esc(label)}</div>'
                f'<div class="btrack"><div class="bfill" style="width:{value / top * 100:.0f}%;background:{color};"></div></div>'
                f'<div class="bcount">{value}</div></div>')
    return out


def load_csv(file):
    raw = file.getvalue()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError("unreadable")


@st.cache_resource
def logo_html():
    """The logo twice (dark letters and light letters); the page shows the one that suits the theme."""
    def encode(path):
        with open(path, "rb") as fh:
            return base64.b64encode(fh.read()).decode()
    if not os.path.exists("logo_clean.png"):
        return "<h1>CheckKro</h1>"
    light = encode("logo_clean.png")
    dark = encode("logo_dark.png") if os.path.exists("logo_dark.png") else light
    return (f'<img class="logo logo-light" src="data:image/png;base64,{light}" alt="CheckKro">'
            f'<img class="logo logo-dark" src="data:image/png;base64,{dark}" alt="CheckKro">')


@st.cache_data(show_spinner="Checking your file...")
def analyse(df):
    findings = checks.run_checkup(df)
    cleaned, log = cleaner.clean(df)
    after = checks.run_checkup(cleaned)
    return findings, cleaned, log, after


# ---------- header and upload ----------
st.markdown(logo_html(), unsafe_allow_html=True)
st.markdown('<div class="tagline">Know if your data is safe to analyze.</div>', unsafe_allow_html=True)

uploaded = st.file_uploader("Upload a CSV file", type="csv")
if st.button("Try with a sample messy file"):
    st.session_state["sample"] = True
if uploaded is not None:
    st.session_state["sample"] = False

df = None
if uploaded is not None:
    try:
        df, enc = load_csv(uploaded)
    except Exception:
        st.error("This file could not be read as a CSV. Check that it is a plain CSV export and try again.")
        st.stop()
    if enc != "utf-8-sig":
        st.info(f"This file is not UTF-8, so it was read as {enc}. Special characters may look different.")
elif st.session_state.get("sample"):
    df = make_messy()
    st.info("Showing the built-in sample file.")

if df is None:
    st.stop()
if df.shape[0] == 0 or df.shape[1] == 0:
    st.warning("This file has no rows to check. Upload a file that contains data.")
    st.stop()

findings, cleaned, log, after = analyse(df)
score = scoring.trust_score(findings)
verdict = scoring.verdict(score, findings)
n_sev = {s: sum(f.severity == s for f in findings) for s in ("high", "medium", "low")}
n_high = n_sev["high"]
affected_cols = Counter(f.column for f in findings if f.column != "(all columns)")

# ---------- dashboard: metrics ----------
c1, c2, c3, c4 = st.columns(4)
c1.metric("Trust score", f"{score} / 100")
c2.metric("Problems found", len(findings))
c3.metric("Columns affected", len(affected_cols))
c4.metric("Rows x columns", f"{len(df):,} x {df.shape[1]}")

# ---------- verdict card with reason ----------
if not findings:
    reason = "No problems found."
else:
    lead = ("No serious issues." if n_high == 0 else
            "1 serious issue requires review." if n_high == 1 else
            f"{n_high} serious issues require review.")
    top = findings[0]
    reason = f"{esc(lead)} Biggest: {esc(title_of(top.check))} in <b>{esc(top.column)}</b>."
st.markdown(
    f'<div class="verdict-card" style="background:{VERDICT_STYLE[verdict]};">'
    f'<div><div class="label">Verdict</div><div class="big">{esc(verdict)}</div></div>'
    f'<div class="reason">{reason}</div></div>',
    unsafe_allow_html=True,
)

# ---------- dashboard: breakdown panels ----------
left, right = st.columns(2)
with left:
    rows = [("High", n_sev["high"], "#D6453F"), ("Medium", n_sev["medium"], "#E08A1E"),
            ("Low", n_sev["low"], "#8CA0BF")]
    st.markdown(f'<div class="panel"><div class="ptitle">Findings by severity</div>{bars(rows, None)}</div>',
                unsafe_allow_html=True)
with right:
    top_cols = [(c, n, "#0F9AA8") for c, n in affected_cols.most_common(5)]
    body = bars(top_cols, None) if top_cols else '<div class="frow">No column has problems.</div>'
    st.markdown(f'<div class="panel"><div class="ptitle">Columns needing attention</div>{body}</div>',
                unsafe_allow_html=True)

tab_findings, tab_score, tab_overview, tab_charts, tab_clean = st.tabs(
    ["Findings", "Score explained", "Overview", "Charts", "Clean the file"])

# ---------- findings ----------
with tab_findings:
    show = st.radio("Show", ["All", "High", "Medium", "Low"], horizontal=True, label_visibility="collapsed")
    shown = [f for f in findings if show == "All" or f.severity == show.lower()]
    if not findings:
        st.success("No problems found.")
    elif not shown:
        st.info("No findings at this severity.")
    for f in shown:
        css, label = SEV[f.severity]
        st.markdown(
            f'<div class="finding"><div class="fhead">'
            f'<span class="pill pill-{css}">{label}</span>'
            f'<span class="ftitle">{esc(title_of(f.check))}</span>'
            f'<span class="fcol">{esc(f.column)}</span>'
            f'<span class="faff">{esc(affected(f))}</span></div>'
            f'<div class="frow"><b>Why it matters:</b> {bold_cols(f.why, df.columns)}</div>'
            f'<div class="frow"><b>Suggested action:</b> {bold_cols(f.suggestion, df.columns)}</div></div>',
            unsafe_allow_html=True,
        )
        with st.expander("Details and impact on your numbers"):
            st.markdown(f'<div class="frow">{bold_cols(f.detail, df.columns)}</div>', unsafe_allow_html=True)
            effect = impact.impact(df, f)
            if effect:
                st.markdown(f'<div class="frow"><b>Impact on your numbers:</b> {bold_cols(effect, df.columns)}</div>',
                            unsafe_allow_html=True)

# ---------- score explained ----------
with tab_score:
    pen = scoring.PENALTY
    st.markdown("**How the score is calculated.** The score starts at 100. Every finding takes points off, "
                "depending on its severity:")
    for s in ("high", "medium", "low"):
        st.markdown(f"- {n_sev[s]} {s} finding(s) x {pen[s]} points = **-{n_sev[s] * pen[s]}**")
    st.markdown(f"Result: **{score} / 100**.")
    costly = sorted(findings, key=lambda f: -pen[f.severity])[:5]
    if costly:
        st.markdown("**What lowers it most:**")
        for f in costly:
            st.markdown(f"- -{pen[f.severity]}: {title_of(f.check)} in **{f.column}**")
    st.markdown("**What the verdict means:**")
    st.markdown("- **Safe to analyze:** score of 85 or more, and no high-severity findings.\n"
                "- **Use with caution:** score of 60 or more.\n"
                "- **Do not use until fixed:** anything below 60.")
    st.caption("The score is a simple rule-based heuristic: fixed penalties by severity. It is not a statistical "
               "probability and does not guarantee that the data is right or wrong. One high-severity finding "
               "always blocks 'safe to analyze'.")

# ---------- overview ----------
with tab_overview:
    ov = checks.overview(df)
    ov["example"] = ov["example"].astype(str)
    st.subheader("Columns")
    st.markdown(table_html(ov), unsafe_allow_html=True)
    cov = checks.date_coverage(df)
    if len(cov) > 0:
        st.subheader("Date coverage")
        st.markdown(table_html(cov), unsafe_allow_html=True)

# ---------- charts ----------
with tab_charts:
    charts.render_charts(df, cleaned)

# ---------- clean ----------
with tab_clean:
    st.write("Only safe fixes are applied: trimming spaces, turning placeholders like 'N/A' "
             "into real blanks, and removing exact duplicate rows. Everything else needs your decision. "
             "Your original file is never changed.")
    placeholders = int(log.loc[log["action"].str.startswith("placeholder"), "rows_affected"].sum()) if len(log) else 0
    after_score = scoring.trust_score(after)
    m1, m2, m3 = st.columns(3)
    m1.metric("Rows", f"{len(df):,} -> {len(cleaned):,}")
    m2.metric("Placeholders turned blank", f"{placeholders:,}")
    m3.metric("Score after safe fixes", f"{after_score} / 100", delta=after_score - score)
    if len(log) == 0:
        st.info("Nothing needed the safe fixes.")
    else:
        st.subheader("Change log")
        st.markdown(table_html(log), unsafe_allow_html=True)
    st.subheader("Preview of the cleaned data")
    st.dataframe(cleaned.head(50), use_container_width=True)
    d1, d2 = st.columns(2)
    d1.download_button("Download cleaned CSV", cleaned.to_csv(index=False).encode("utf-8"),
                       file_name="cleaned.csv", mime="text/csv")
    d2.download_button("Download change log (CSV)", log.to_csv(index=False).encode("utf-8"),
                       file_name="change_log.csv", mime="text/csv")

st.caption("CheckKro uses rule-based checks. Nothing in your file is changed without a logged record.")

Overwriting app.py


**Light and dark mode with a button.** A round button fixed in the top right corner switches the whole app between light and dark. All colors are defined once at the top, one set for each mode, so the page, cards, tables, tags, tabs, buttons, charts and the logo switch together. Light mode uses a teal accent, and dark mode uses near-black with a sky blue accent. Streamlit's own inputs and dropdowns are restyled so they follow the button, and the cleaned-data preview uses our own table so it can be styled too.

In [42]:
%%writefile app.py
import base64
import html
import io
import os
from collections import Counter

import pandas as pd
import streamlit as st

import charts
import checks
import cleaner
import impact
import scoring
from datagen import make_messy

st.set_page_config(page_title="CheckKro", page_icon="✅", layout="wide")

# ---------- light / dark mode (the button in the top right corner) ----------
LIGHT = """
    color-scheme: light;
    --bg1: #EEF4F8; --bg2: #DDE9F4;
    --card: rgba(255,255,255,0.80); --card-solid: #FFFFFF; --card-line: rgba(255,255,255,0.95);
    --input-bg: #FFFFFF;
    --text: #0B1B3A; --body: #1F2A3C; --muted: #5B6B85;
    --accent: #0F9AA8; --accent-hover: #0B7F8B; --on-accent: #FFFFFF;
    --accent-soft: #DDF1F3; --accent-text: #0F7F8B;
    --track: #E6EEF9; --line: #D5E1F5; --shadow: rgba(40,80,160,0.09);
    --tab: rgba(255,255,255,0.65);
    --hi-fg: #B42318; --hi-bg: #FDECEA; --me-fg: #B45309; --me-bg: #FEF3E2; --lo-fg: #4B5B78; --lo-bg: #EAF0F8;
"""
DARK = """
    color-scheme: dark;
    --bg1: #050506; --bg2: #0A0C10;
    --card: rgba(15,16,19,0.92); --card-solid: #0F1013; --card-line: rgba(255,255,255,0.06);
    --input-bg: #14161B;
    --text: #F2F4F8; --body: #C9CFDA; --muted: #9AA1AD;
    --accent: #38BDF8; --accent-hover: #7DD3FC; --on-accent: #04202C;
    --accent-soft: #0B2230; --accent-text: #7DD3FC;
    --track: #1A1D23; --line: #22252B; --shadow: rgba(0,0,0,0.6);
    --tab: rgba(15,16,19,0.7);
    --hi-fg: #F0716B; --hi-bg: #2A1316; --me-fg: #F5B54A; --me-bg: #2A210E; --lo-fg: #9FB0CC; --lo-bg: #1A1D23;
"""


def flip_theme():
    st.session_state["theme"] = "dark" if st.session_state.get("theme", "light") == "light" else "light"


mode = st.session_state.get("theme", "light")
st.button("🌙" if mode == "light" else "☀️", key="theme_toggle", on_click=flip_theme,
          help="Switch between light and dark mode")
st.markdown(f"<style>:root {{{DARK if mode == 'dark' else LIGHT}}}</style>", unsafe_allow_html=True)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.stApp, .stApp p, .stApp label, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4,
.stApp button, .stApp input, .stApp textarea, .tagline, .verdict-card, .verdict-card div,
.panel, .panel div, .finding, .finding div, .finding span,
[data-testid="stMetricValue"], [data-testid="stMetricLabel"] {
    font-family: 'Inter', -apple-system, 'Segoe UI', Roboto, sans-serif !important;
}

.stApp {background: linear-gradient(160deg, var(--bg1) 0%, var(--bg2) 100%);}
header[data-testid="stHeader"] {background: transparent;}
[data-testid="stToolbar"], [data-testid="stDecoration"], #MainMenu, footer {display: none !important;}
.block-container {padding-top: 2rem; max-width: 1150px;}

.stApp p, .stApp li {font-size: 0.95rem; line-height: 1.65; color: var(--body);}
.stApp h1, .stApp h2, .stApp h3 {font-weight: 600; letter-spacing: -0.01em; color: var(--text);}
.stApp [data-testid="stCaptionContainer"] p {color: var(--muted);}
.tagline {color: var(--muted); font-size: 1.05rem; margin: 0.3rem 0 1.4rem;}
.logo {width: 230px; display: block;}

[data-testid="stMetricLabel"] p {font-size: 0.78rem; font-weight: 500; letter-spacing: 0.06em;
                                 text-transform: uppercase; color: var(--muted);}
[data-testid="stMetricValue"] {font-size: 1.8rem; color: var(--text); font-weight: 700;}
[data-testid="stMetric"] {
    background: var(--card); border: 1px solid var(--card-line);
    border-top: 3px solid var(--accent); border-radius: 18px; padding: 16px 20px;
    box-shadow: 0 6px 24px var(--shadow);
}

.verdict-card {display: flex; justify-content: space-between; align-items: center; gap: 24px;
               border-radius: 18px; padding: 16px 24px; margin: 14px 0 18px; color: #FFFFFF;
               box-shadow: 0 8px 24px var(--shadow);}
.verdict-card .label {font-size: 0.72rem; letter-spacing: 0.1em; text-transform: uppercase; opacity: 0.85;}
.verdict-card .big {font-size: 1.35rem; font-weight: 800;}
.verdict-card .reason {text-align: right; max-width: 55%; font-size: 0.95rem; opacity: 0.97;}
.verdict-card .reason b {font-weight: 800;}

.panel {background: var(--card); border: 1px solid var(--card-line);
        border-radius: 18px; padding: 16px 20px; box-shadow: 0 6px 24px var(--shadow);
        margin-bottom: 14px;}
.ptitle {font-weight: 600; color: var(--text); margin-bottom: 8px;}
.brow {display: flex; align-items: center; gap: 10px; margin: 8px 0; font-size: 0.9rem; color: var(--body);}
.blabel {width: 140px; flex: none; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
         font-weight: 700; color: var(--text);}
.btrack {flex: 1; background: var(--track); border-radius: 99px; height: 9px;}
.bfill {height: 9px; border-radius: 99px;}
.bcount {width: 30px; text-align: right; color: var(--muted);}

.finding {background: var(--card); border: 1px solid var(--card-line);
          border-radius: 16px; padding: 14px 18px; margin: 12px 0 4px;
          box-shadow: 0 4px 18px var(--shadow);}
.fhead {display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin-bottom: 6px;}
.pill {font-size: 0.7rem; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase;
       padding: 3px 10px; border-radius: 99px;}
.pill-hi {color: var(--hi-fg); background: var(--hi-bg);}
.pill-me {color: var(--me-fg); background: var(--me-bg);}
.pill-lo {color: var(--lo-fg); background: var(--lo-bg);}
.ftitle {font-weight: 600; color: var(--text);}
.fcol {color: var(--text); font-weight: 700; font-family: ui-monospace, Menlo, monospace; font-size: 0.85rem;
       background: var(--accent-soft); padding: 2px 9px; border-radius: 7px;}
.faff {margin-left: auto; color: var(--muted); font-size: 0.88rem;}
.frow {font-size: 0.92rem; color: var(--body); margin: 3px 0;}
.frow b {color: var(--text); font-weight: 700;}

.tbl {overflow-x: auto; background: var(--card); border: 1px solid var(--card-line);
      border-radius: 16px; padding: 6px 12px; margin-bottom: 14px;}
.tbl table {border-collapse: collapse; width: 100%; font-size: 0.88rem;}
.tbl th {text-align: left; color: var(--muted); font-size: 0.72rem; text-transform: uppercase;
         letter-spacing: 0.06em; padding: 9px 10px; border-bottom: 1px solid var(--line); white-space: nowrap;}
.tbl td {padding: 8px 10px; border-bottom: 1px solid var(--line); color: var(--body);}
.tbl td.cn {font-weight: 700; color: var(--text);}

div[data-baseweb="tab-list"] {gap: 8px; background: transparent;}
button[data-baseweb="tab"] {background: var(--tab); border: 1px solid var(--line);
                            border-radius: 999px; padding: 6px 18px; height: auto;}
button[data-baseweb="tab"][aria-selected="true"] {background: var(--accent); border-color: var(--accent);}
button[data-baseweb="tab"][aria-selected="true"] p {color: var(--on-accent);}
div[data-baseweb="tab-highlight"], div[data-baseweb="tab-border"] {display: none;}

[data-testid="stExpander"] {background: var(--card); border: 1px solid var(--card-line);
                            border-radius: 14px; overflow: hidden;}
[data-testid="stExpander"] details {border: none;}
[data-testid="stExpander"] summary p {font-size: 0.9rem; font-weight: 500; color: var(--accent-text);}
[data-testid="stExpander"] summary svg {color: var(--accent-text);}

.stButton > button, .stDownloadButton > button {background: var(--accent); color: var(--on-accent); border: none;
                                                border-radius: 12px; padding: 0.55rem 1.3rem;}
.stButton > button p, .stDownloadButton > button p {color: var(--on-accent);}
.stButton > button:hover, .stDownloadButton > button:hover {background: var(--accent-hover);}

/* the light / dark button, fixed in the top right corner */
.st-key-theme_toggle {position: fixed; top: 0.9rem; right: 1.1rem; z-index: 1000000; width: auto !important;}
.st-key-theme_toggle button {width: 44px; height: 44px; min-height: 44px; padding: 0; border-radius: 50%;
                             background: var(--card-solid); border: 1px solid var(--line);
                             box-shadow: 0 6px 18px var(--shadow);}
.st-key-theme_toggle button p {color: var(--text); font-size: 1.15rem; line-height: 1; margin: 0;}
.st-key-theme_toggle button:hover {background: var(--accent-soft); border-color: var(--accent);}

/* Streamlit's own widgets, restyled so they follow the button and not the device */
[data-testid="stWidgetLabel"] p {color: var(--muted);}
[data-baseweb="input"], [data-baseweb="base-input"], [data-baseweb="textarea"], [data-baseweb="select"] > div {
    background: var(--input-bg) !important; border-color: var(--line) !important; color: var(--text) !important;
}
[data-baseweb="input"] input, [data-baseweb="select"] input, [data-baseweb="select"] div {color: var(--text) !important;}
[data-baseweb="select"] svg {fill: var(--muted) !important;}
[data-baseweb="popover"] > div, [data-baseweb="popover"] ul, [role="listbox"] {
    background: var(--card-solid) !important; color: var(--body) !important;
}
[data-baseweb="popover"] li, [role="option"] {color: var(--body) !important; background: transparent !important;}
[data-baseweb="popover"] li:hover, [role="option"]:hover, [role="option"][aria-selected="true"] {
    background: var(--accent-soft) !important;
}
[data-testid="stSliderTickBarMin"], [data-testid="stSliderTickBarMax"] {color: var(--muted);}
[data-testid="stThumbValue"] {color: var(--text);}
[data-testid="stSlider"] [role="slider"] {background-color: var(--accent) !important;}

[data-testid="stFileUploader"] section {background: var(--card); border: 1.5px dashed var(--accent);
                                        border-radius: 18px;}
[data-testid="stFileUploader"] small, [data-testid="stFileUploader"] section span,
[data-testid="stFileUploaderFile"] {color: var(--muted);}
[data-testid="stFileUploaderDropzone"] button {background: var(--accent); color: var(--on-accent); border: none;
                                               border-radius: 10px;}
[data-testid="stFileUploaderDropzone"] button p {color: var(--on-accent);}

[data-testid="stAlert"], [data-testid="stAlert"] > div {
    background: var(--card-solid) !important; border-radius: 14px; color: var(--body);
}
[data-testid="stAlert"] {border: 1px solid var(--line);}
[data-testid="stAlert"] p {color: var(--body) !important;}

/* ---------- mobile view ---------- */
@media (max-width: 640px) {
    .block-container {padding: 1rem 0.9rem 3rem;}
    .tagline {font-size: 0.95rem; margin-bottom: 1rem;}
    .logo {width: 180px;}

    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) {flex-wrap: wrap; gap: 0.6rem;}
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="stColumn"],
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="column"] {
        flex: 1 1 calc(50% - 0.6rem) !important;
        min-width: calc(50% - 0.6rem) !important;
        width: calc(50% - 0.6rem) !important;
    }
    [data-testid="stMetric"] {padding: 12px 14px;}
    [data-testid="stMetricValue"] {font-size: 1.3rem;}

    .verdict-card {flex-direction: column; align-items: flex-start; gap: 8px; padding: 14px 18px;}
    .verdict-card .reason {text-align: left; max-width: 100%;}
    .verdict-card .big {font-size: 1.15rem;}

    .panel {padding: 14px 16px;}
    .blabel {width: 96px;}
    .finding {padding: 12px 14px;}
    .faff {margin-left: 0; width: 100%;}
    button[data-baseweb="tab"] {padding: 5px 12px;}
    .stButton > button, .stDownloadButton > button {width: 100%;}
    .st-key-theme_toggle button {width: 44px;}
}
</style>
""", unsafe_allow_html=True)

TITLES = {
    "missing_values": "Missing values",
    "structural_missing": "Blank by design",
    "disguised_missing": "Placeholder values (like N/A)",
    "exact_duplicates": "Duplicate rows",
    "duplicate_ids": "Repeated IDs",
    "extra_whitespace": "Extra spaces",
    "inconsistent_case": "Inconsistent spelling or case",
    "numeric_as_text": "Numbers stored as text",
    "mixed_types": "Text mixed into numbers",
    "impossible_values": "Impossible values",
    "extreme_outliers": "Extreme outliers",
    "future_dates": "Dates in the future",
    "date_order": "Dates in the wrong order",
    "suspicious_rate": "Suspiciously high rate",
    "constant_column": "Column never changes",
    "all_unique_column": "Every value is unique",
    "messy_column_name": "Messy column name",
    "date_gaps": "Missing days in the data",
    "rare_categories": "Rare values (possible typos)",
}
SEV = {"high": ("hi", "High"), "medium": ("me", "Medium"), "low": ("lo", "Low")}
VERDICT_STYLE = {
    "SAFE TO ANALYZE": "linear-gradient(135deg, #27B5A5, #0F8F97)",
    "USE WITH CAUTION": "linear-gradient(135deg, #F5B54A, #E08A1E)",
    "DO NOT USE UNTIL FIXED": "linear-gradient(135deg, #F0716B, #D6453F)",
}


def esc(x):
    return html.escape(str(x))


def bold_cols(text, columns):
    """Escape text and make any 'column name' in quotes bold."""
    out = html.escape(str(text))
    for c in sorted(map(str, columns), key=len, reverse=True):
        out = out.replace(html.escape(f"'{c}'"), f"<b>{html.escape(c)}</b>")
    return out


def table_html(frame):
    """A simple styled table where the 'column' column is bold."""
    head = "".join(f"<th>{esc(c)}</th>" for c in frame.columns)
    body = ""
    for _, row in frame.iterrows():
        cells = "".join(
            f'<td class="cn">{esc(v)}</td>' if c == "column" else f"<td>{esc(v)}</td>"
            for c, v in zip(frame.columns, row.values))
        body += f"<tr>{cells}</tr>"
    return f'<div class="tbl"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def title_of(check):
    return TITLES.get(check, check.replace("_", " ").capitalize())


def affected(f):
    return f"{f.count:,} rows ({f.pct}%)" if f.count else "Column-level issue"


def bars(rows, colors):
    """rows: list of (label, value, color). Draws simple horizontal bars."""
    top = max([v for _, v, _ in rows] + [1])
    out = ""
    for label, value, color in rows:
        out += (f'<div class="brow"><div class="blabel">{esc(label)}</div>'
                f'<div class="btrack"><div class="bfill" style="width:{value / top * 100:.0f}%;background:{color};"></div></div>'
                f'<div class="bcount">{value}</div></div>')
    return out


def load_csv(file):
    raw = file.getvalue()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError("unreadable")


@st.cache_resource
def logo_html(theme):
    """The logo with dark letters for light mode, and light letters for dark mode."""
    def encode(path):
        with open(path, "rb") as fh:
            return base64.b64encode(fh.read()).decode()
    if not os.path.exists("logo_clean.png"):
        return "<h1>CheckKro</h1>"
    path = "logo_dark.png" if theme == "dark" and os.path.exists("logo_dark.png") else "logo_clean.png"
    return f'<img class="logo" src="data:image/png;base64,{encode(path)}" alt="CheckKro">'


@st.cache_data(show_spinner="Checking your file...")
def analyse(df):
    findings = checks.run_checkup(df)
    cleaned, log = cleaner.clean(df)
    after = checks.run_checkup(cleaned)
    return findings, cleaned, log, after


# ---------- header and upload ----------
st.markdown(logo_html(mode), unsafe_allow_html=True)
st.markdown('<div class="tagline">Know if your data is safe to analyze.</div>', unsafe_allow_html=True)

uploaded = st.file_uploader("Upload a CSV file", type="csv")
if st.button("Try with a sample messy file"):
    st.session_state["sample"] = True
if uploaded is not None:
    st.session_state["sample"] = False

df = None
if uploaded is not None:
    try:
        df, enc = load_csv(uploaded)
    except Exception:
        st.error("This file could not be read as a CSV. Check that it is a plain CSV export and try again.")
        st.stop()
    if enc != "utf-8-sig":
        st.info(f"This file is not UTF-8, so it was read as {enc}. Special characters may look different.")
elif st.session_state.get("sample"):
    df = make_messy()
    st.info("Showing the built-in sample file.")

if df is None:
    st.stop()
if df.shape[0] == 0 or df.shape[1] == 0:
    st.warning("This file has no rows to check. Upload a file that contains data.")
    st.stop()

findings, cleaned, log, after = analyse(df)
score = scoring.trust_score(findings)
verdict = scoring.verdict(score, findings)
n_sev = {s: sum(f.severity == s for f in findings) for s in ("high", "medium", "low")}
n_high = n_sev["high"]
affected_cols = Counter(f.column for f in findings if f.column != "(all columns)")

# ---------- dashboard: metrics ----------
c1, c2, c3, c4 = st.columns(4)
c1.metric("Trust score", f"{score} / 100")
c2.metric("Problems found", len(findings))
c3.metric("Columns affected", len(affected_cols))
c4.metric("Rows x columns", f"{len(df):,} x {df.shape[1]}")

# ---------- verdict card with reason ----------
if not findings:
    reason = "No problems found."
else:
    lead = ("No serious issues." if n_high == 0 else
            "1 serious issue requires review." if n_high == 1 else
            f"{n_high} serious issues require review.")
    top = findings[0]
    reason = f"{esc(lead)} Biggest: {esc(title_of(top.check))} in <b>{esc(top.column)}</b>."
st.markdown(
    f'<div class="verdict-card" style="background:{VERDICT_STYLE[verdict]};">'
    f'<div><div class="label">Verdict</div><div class="big">{esc(verdict)}</div></div>'
    f'<div class="reason">{reason}</div></div>',
    unsafe_allow_html=True,
)

# ---------- dashboard: breakdown panels ----------
left, right = st.columns(2)
with left:
    rows = [("High", n_sev["high"], "#D6453F"), ("Medium", n_sev["medium"], "#E08A1E"),
            ("Low", n_sev["low"], "#8CA0BF")]
    st.markdown(f'<div class="panel"><div class="ptitle">Findings by severity</div>{bars(rows, None)}</div>',
                unsafe_allow_html=True)
with right:
    top_cols = [(c, n, "var(--accent)") for c, n in affected_cols.most_common(5)]
    body = bars(top_cols, None) if top_cols else '<div class="frow">No column has problems.</div>'
    st.markdown(f'<div class="panel"><div class="ptitle">Columns needing attention</div>{body}</div>',
                unsafe_allow_html=True)

tab_findings, tab_score, tab_overview, tab_charts, tab_clean = st.tabs(
    ["Findings", "Score explained", "Overview", "Charts", "Clean the file"])

# ---------- findings ----------
with tab_findings:
    show = st.radio("Show", ["All", "High", "Medium", "Low"], horizontal=True, label_visibility="collapsed")
    shown = [f for f in findings if show == "All" or f.severity == show.lower()]
    if not findings:
        st.success("No problems found.")
    elif not shown:
        st.info("No findings at this severity.")
    for f in shown:
        css, label = SEV[f.severity]
        st.markdown(
            f'<div class="finding"><div class="fhead">'
            f'<span class="pill pill-{css}">{label}</span>'
            f'<span class="ftitle">{esc(title_of(f.check))}</span>'
            f'<span class="fcol">{esc(f.column)}</span>'
            f'<span class="faff">{esc(affected(f))}</span></div>'
            f'<div class="frow"><b>Why it matters:</b> {bold_cols(f.why, df.columns)}</div>'
            f'<div class="frow"><b>Suggested action:</b> {bold_cols(f.suggestion, df.columns)}</div></div>',
            unsafe_allow_html=True,
        )
        with st.expander("Details and impact on your numbers"):
            st.markdown(f'<div class="frow">{bold_cols(f.detail, df.columns)}</div>', unsafe_allow_html=True)
            effect = impact.impact(df, f)
            if effect:
                st.markdown(f'<div class="frow"><b>Impact on your numbers:</b> {bold_cols(effect, df.columns)}</div>',
                            unsafe_allow_html=True)

# ---------- score explained ----------
with tab_score:
    pen = scoring.PENALTY
    st.markdown("**How the score is calculated.** The score starts at 100. Every finding takes points off, "
                "depending on its severity:")
    for s in ("high", "medium", "low"):
        st.markdown(f"- {n_sev[s]} {s} finding(s) x {pen[s]} points = **-{n_sev[s] * pen[s]}**")
    st.markdown(f"Result: **{score} / 100**.")
    costly = sorted(findings, key=lambda f: -pen[f.severity])[:5]
    if costly:
        st.markdown("**What lowers it most:**")
        for f in costly:
            st.markdown(f"- -{pen[f.severity]}: {title_of(f.check)} in **{f.column}**")
    st.markdown("**What the verdict means:**")
    st.markdown("- **Safe to analyze:** score of 85 or more, and no high-severity findings.\n"
                "- **Use with caution:** score of 60 or more.\n"
                "- **Do not use until fixed:** anything below 60.")
    st.caption("The score is a simple rule-based heuristic: fixed penalties by severity. It is not a statistical "
               "probability and does not guarantee that the data is right or wrong. One high-severity finding "
               "always blocks 'safe to analyze'.")

# ---------- overview ----------
with tab_overview:
    ov = checks.overview(df)
    ov["example"] = ov["example"].astype(str)
    st.subheader("Columns")
    st.markdown(table_html(ov), unsafe_allow_html=True)
    cov = checks.date_coverage(df)
    if len(cov) > 0:
        st.subheader("Date coverage")
        st.markdown(table_html(cov), unsafe_allow_html=True)

# ---------- charts ----------
with tab_charts:
    charts.TEAL = "#38BDF8" if mode == "dark" else "#0F9AA8"
    charts.render_charts(df, cleaned)

# ---------- clean ----------
with tab_clean:
    st.write("Only safe fixes are applied: trimming spaces, turning placeholders like 'N/A' "
             "into real blanks, and removing exact duplicate rows. Everything else needs your decision. "
             "Your original file is never changed.")
    placeholders = int(log.loc[log["action"].str.startswith("placeholder"), "rows_affected"].sum()) if len(log) else 0
    after_score = scoring.trust_score(after)
    m1, m2, m3 = st.columns(3)
    m1.metric("Rows", f"{len(df):,} -> {len(cleaned):,}")
    m2.metric("Placeholders turned blank", f"{placeholders:,}")
    m3.metric("Score after safe fixes", f"{after_score} / 100", delta=after_score - score)
    if len(log) == 0:
        st.info("Nothing needed the safe fixes.")
    else:
        st.subheader("Change log")
        st.markdown(table_html(log), unsafe_allow_html=True)
    st.subheader("Preview of the cleaned data")
    preview = cleaned.head(50).astype(object).where(lambda x: x.notna(), "")
    st.markdown(table_html(preview), unsafe_allow_html=True)
    d1, d2 = st.columns(2)
    d1.download_button("Download cleaned CSV", cleaned.to_csv(index=False).encode("utf-8"),
                       file_name="cleaned.csv", mime="text/csv")
    d2.download_button("Download change log (CSV)", log.to_csv(index=False).encode("utf-8"),
                       file_name="change_log.csv", mime="text/csv")

st.caption("CheckKro uses rule-based checks. Nothing in your file is changed without a logged record.")

Overwriting app.py


**Make the result easier to read.** The trust score card now has a bar that fills to the score and changes color with the verdict (green for safe, amber for caution, red for do not use), with marks at 60 and 85 and a short note. The verdict card says how many issues need attention and lists each one in a short plain sentence, with the column and the key words in bold. The sample-file button is hidden once a file is uploaded.

In [43]:
%%writefile app.py
import base64
import html
import io
import os
from collections import Counter

import pandas as pd
import streamlit as st

import charts
import checks
import cleaner
import impact
import scoring
from datagen import make_messy

st.set_page_config(page_title="CheckKro", page_icon="✅", layout="wide")

# ---------- light / dark mode (the button in the top right corner) ----------
LIGHT = """
    color-scheme: light;
    --bg1: #EEF4F8; --bg2: #DDE9F4;
    --card: rgba(255,255,255,0.80); --card-solid: #FFFFFF; --card-line: rgba(255,255,255,0.95);
    --input-bg: #FFFFFF;
    --text: #0B1B3A; --body: #1F2A3C; --muted: #5B6B85;
    --accent: #0F9AA8; --accent-hover: #0B7F8B; --on-accent: #FFFFFF;
    --accent-soft: #DDF1F3; --accent-text: #0F7F8B;
    --track: #E6EEF9; --line: #D5E1F5; --shadow: rgba(40,80,160,0.09);
    --tab: rgba(255,255,255,0.65);
    --hi-fg: #B42318; --hi-bg: #FDECEA; --me-fg: #B45309; --me-bg: #FEF3E2; --lo-fg: #4B5B78; --lo-bg: #EAF0F8;
"""
DARK = """
    color-scheme: dark;
    --bg1: #050506; --bg2: #0A0C10;
    --card: rgba(15,16,19,0.92); --card-solid: #0F1013; --card-line: rgba(255,255,255,0.06);
    --input-bg: #14161B;
    --text: #F2F4F8; --body: #C9CFDA; --muted: #9AA1AD;
    --accent: #38BDF8; --accent-hover: #7DD3FC; --on-accent: #04202C;
    --accent-soft: #0B2230; --accent-text: #7DD3FC;
    --track: #1A1D23; --line: #22252B; --shadow: rgba(0,0,0,0.6);
    --tab: rgba(15,16,19,0.7);
    --hi-fg: #F0716B; --hi-bg: #2A1316; --me-fg: #F5B54A; --me-bg: #2A210E; --lo-fg: #9FB0CC; --lo-bg: #1A1D23;
"""


def flip_theme():
    st.session_state["theme"] = "dark" if st.session_state.get("theme", "light") == "light" else "light"


mode = st.session_state.get("theme", "light")
st.button("🌙" if mode == "light" else "☀️", key="theme_toggle", on_click=flip_theme,
          help="Switch between light and dark mode")
st.markdown(f"<style>:root {{{DARK if mode == 'dark' else LIGHT}}}</style>", unsafe_allow_html=True)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.stApp, .stApp p, .stApp label, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4,
.stApp button, .stApp input, .stApp textarea, .tagline, .verdict-card, .verdict-card div,
.panel, .panel div, .finding, .finding div, .finding span, .scorecard, .scorecard div,
[data-testid="stMetricValue"], [data-testid="stMetricLabel"] {
    font-family: 'Inter', -apple-system, 'Segoe UI', Roboto, sans-serif !important;
}

.stApp {background: linear-gradient(160deg, var(--bg1) 0%, var(--bg2) 100%);}
header[data-testid="stHeader"] {background: transparent;}
[data-testid="stToolbar"], [data-testid="stDecoration"], #MainMenu, footer {display: none !important;}
.block-container {padding-top: 2rem; max-width: 1150px;}

.stApp p, .stApp li {font-size: 0.95rem; line-height: 1.65; color: var(--body);}
.stApp h1, .stApp h2, .stApp h3 {font-weight: 600; letter-spacing: -0.01em; color: var(--text);}
.stApp [data-testid="stCaptionContainer"] p {color: var(--muted);}
.tagline {color: var(--muted); font-size: 1.05rem; margin: 0.3rem 0 1.4rem;}
.logo {width: 230px; display: block;}

[data-testid="stMetricLabel"] p {font-size: 0.78rem; font-weight: 500; letter-spacing: 0.06em;
                                 text-transform: uppercase; color: var(--muted);}
[data-testid="stMetricValue"] {font-size: 1.8rem; color: var(--text); font-weight: 700;}
[data-testid="stMetric"], .scorecard {
    background: var(--card); border: 1px solid var(--card-line);
    border-top: 3px solid var(--accent); border-radius: 18px; padding: 16px 20px;
    box-shadow: 0 6px 24px var(--shadow); min-height: 132px; box-sizing: border-box;
}
.scorecard .slabel {font-size: 0.78rem; font-weight: 500; letter-spacing: 0.06em; text-transform: uppercase;
                    color: var(--muted);}
.scorecard .svalue {font-size: 1.8rem; font-weight: 700; color: var(--text); line-height: 1.3;}
.sbar {position: relative; height: 10px; background: var(--track); border-radius: 99px; margin: 8px 0 7px;}
.sfill {height: 100%; border-radius: 99px;}
.stick {position: absolute; top: -3px; bottom: -3px; width: 2px; background: var(--muted); opacity: 0.55;}
.scorecard .snote {font-size: 0.78rem; color: var(--muted); line-height: 1.35;}

.verdict-card {display: flex; justify-content: space-between; align-items: center; gap: 24px;
               border-radius: 18px; padding: 16px 24px; margin: 14px 0 18px; color: #FFFFFF;
               box-shadow: 0 8px 24px var(--shadow);}
.verdict-card .label {font-size: 0.72rem; letter-spacing: 0.1em; text-transform: uppercase; opacity: 0.85;}
.verdict-card .big {font-size: 1.35rem; font-weight: 800;}
.verdict-card .reason {text-align: left; max-width: 60%; font-size: 0.95rem;}
.verdict-card .reason b {font-weight: 800;}
.verdict-card .rhead {font-weight: 700; font-size: 1.02rem; margin-bottom: 4px;}
.verdict-card ul {margin: 0; padding-left: 1.1rem;}
.verdict-card ul li {color: #FFFFFF; font-size: 0.93rem; line-height: 1.5; margin: 2px 0;}
.verdict-card .rmore {opacity: 0.85; font-size: 0.85rem; margin-top: 4px;}

.panel {background: var(--card); border: 1px solid var(--card-line);
        border-radius: 18px; padding: 16px 20px; box-shadow: 0 6px 24px var(--shadow);
        margin-bottom: 14px;}
.ptitle {font-weight: 600; color: var(--text); margin-bottom: 8px;}
.brow {display: flex; align-items: center; gap: 10px; margin: 8px 0; font-size: 0.9rem; color: var(--body);}
.blabel {width: 140px; flex: none; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
         font-weight: 700; color: var(--text);}
.btrack {flex: 1; background: var(--track); border-radius: 99px; height: 9px;}
.bfill {height: 9px; border-radius: 99px;}
.bcount {width: 30px; text-align: right; color: var(--muted);}

.finding {background: var(--card); border: 1px solid var(--card-line);
          border-radius: 16px; padding: 14px 18px; margin: 12px 0 4px;
          box-shadow: 0 4px 18px var(--shadow);}
.fhead {display: flex; flex-wrap: wrap; align-items: center; gap: 10px; margin-bottom: 6px;}
.pill {font-size: 0.7rem; font-weight: 700; letter-spacing: 0.06em; text-transform: uppercase;
       padding: 3px 10px; border-radius: 99px;}
.pill-hi {color: var(--hi-fg); background: var(--hi-bg);}
.pill-me {color: var(--me-fg); background: var(--me-bg);}
.pill-lo {color: var(--lo-fg); background: var(--lo-bg);}
.ftitle {font-weight: 600; color: var(--text);}
.fcol {color: var(--text); font-weight: 700; font-family: ui-monospace, Menlo, monospace; font-size: 0.85rem;
       background: var(--accent-soft); padding: 2px 9px; border-radius: 7px;}
.faff {margin-left: auto; color: var(--muted); font-size: 0.88rem;}
.frow {font-size: 0.92rem; color: var(--body); margin: 3px 0;}
.frow b {color: var(--text); font-weight: 700;}

.tbl {overflow-x: auto; background: var(--card); border: 1px solid var(--card-line);
      border-radius: 16px; padding: 6px 12px; margin-bottom: 14px;}
.tbl table {border-collapse: collapse; width: 100%; font-size: 0.88rem;}
.tbl th {text-align: left; color: var(--muted); font-size: 0.72rem; text-transform: uppercase;
         letter-spacing: 0.06em; padding: 9px 10px; border-bottom: 1px solid var(--line); white-space: nowrap;}
.tbl td {padding: 8px 10px; border-bottom: 1px solid var(--line); color: var(--body);}
.tbl td.cn {font-weight: 700; color: var(--text);}

div[data-baseweb="tab-list"] {gap: 8px; background: transparent;}
button[data-baseweb="tab"] {background: var(--tab); border: 1px solid var(--line);
                            border-radius: 999px; padding: 6px 18px; height: auto;}
button[data-baseweb="tab"][aria-selected="true"] {background: var(--accent); border-color: var(--accent);}
button[data-baseweb="tab"][aria-selected="true"] p {color: var(--on-accent);}
div[data-baseweb="tab-highlight"], div[data-baseweb="tab-border"] {display: none;}

[data-testid="stExpander"] {background: var(--card); border: 1px solid var(--card-line);
                            border-radius: 14px; overflow: hidden;}
[data-testid="stExpander"] details {border: none;}
[data-testid="stExpander"] summary p {font-size: 0.9rem; font-weight: 500; color: var(--accent-text);}
[data-testid="stExpander"] summary svg {color: var(--accent-text);}

.stButton > button, .stDownloadButton > button {background: var(--accent); color: var(--on-accent); border: none;
                                                border-radius: 12px; padding: 0.55rem 1.3rem;}
.stButton > button p, .stDownloadButton > button p {color: var(--on-accent);}
.stButton > button:hover, .stDownloadButton > button:hover {background: var(--accent-hover);}

/* the light / dark button, fixed in the top right corner */
.st-key-theme_toggle {position: fixed; top: 0.9rem; right: 1.1rem; z-index: 1000000; width: auto !important;}
.st-key-theme_toggle button {width: 44px; height: 44px; min-height: 44px; padding: 0; border-radius: 50%;
                             background: var(--card-solid); border: 1px solid var(--line);
                             box-shadow: 0 6px 18px var(--shadow);}
.st-key-theme_toggle button p {color: var(--text); font-size: 1.15rem; line-height: 1; margin: 0;}
.st-key-theme_toggle button:hover {background: var(--accent-soft); border-color: var(--accent);}

/* Streamlit's own widgets, restyled so they follow the button and not the device */
[data-testid="stWidgetLabel"] p {color: var(--muted);}
[data-baseweb="input"], [data-baseweb="base-input"], [data-baseweb="textarea"], [data-baseweb="select"] > div {
    background: var(--input-bg) !important; border-color: var(--line) !important; color: var(--text) !important;
}
[data-baseweb="input"] input, [data-baseweb="select"] input, [data-baseweb="select"] div {color: var(--text) !important;}
[data-baseweb="select"] svg {fill: var(--muted) !important;}
[data-baseweb="popover"] > div, [data-baseweb="popover"] ul, [role="listbox"] {
    background: var(--card-solid) !important; color: var(--body) !important;
}
[data-baseweb="popover"] li, [role="option"] {color: var(--body) !important; background: transparent !important;}
[data-baseweb="popover"] li:hover, [role="option"]:hover, [role="option"][aria-selected="true"] {
    background: var(--accent-soft) !important;
}
[data-testid="stSliderTickBarMin"], [data-testid="stSliderTickBarMax"] {color: var(--muted);}
[data-testid="stThumbValue"] {color: var(--text);}
[data-testid="stSlider"] [role="slider"] {background-color: var(--accent) !important;}

[data-testid="stFileUploader"] section {background: var(--card); border: 1.5px dashed var(--accent);
                                        border-radius: 18px;}
[data-testid="stFileUploader"] small, [data-testid="stFileUploader"] section span,
[data-testid="stFileUploaderFile"] {color: var(--muted);}
[data-testid="stFileUploaderDropzone"] button {background: var(--accent); color: var(--on-accent); border: none;
                                               border-radius: 10px;}
[data-testid="stFileUploaderDropzone"] button p {color: var(--on-accent);}

[data-testid="stAlert"], [data-testid="stAlert"] > div {
    background: var(--card-solid) !important; border-radius: 14px; color: var(--body);
}
[data-testid="stAlert"] {border: 1px solid var(--line);}
[data-testid="stAlert"] p {color: var(--body) !important;}

/* ---------- mobile view ---------- */
@media (max-width: 640px) {
    .block-container {padding: 1rem 0.9rem 3rem;}
    .tagline {font-size: 0.95rem; margin-bottom: 1rem;}
    .logo {width: 180px;}

    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) {flex-wrap: wrap; gap: 0.6rem;}
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="stColumn"],
    [data-testid="stHorizontalBlock"]:has([data-testid="stMetric"]) > [data-testid="column"] {
        flex: 1 1 calc(50% - 0.6rem) !important;
        min-width: calc(50% - 0.6rem) !important;
        width: calc(50% - 0.6rem) !important;
    }
    [data-testid="stMetric"], .scorecard {padding: 12px 14px; min-height: 0;}
    [data-testid="stMetricValue"] {font-size: 1.3rem;}
    .scorecard .svalue {font-size: 1.3rem;}

    .verdict-card {flex-direction: column; align-items: flex-start; gap: 8px; padding: 14px 18px;}
    .verdict-card .reason {text-align: left; max-width: 100%;}
    .verdict-card .big {font-size: 1.15rem;}

    .panel {padding: 14px 16px;}
    .blabel {width: 96px;}
    .finding {padding: 12px 14px;}
    .faff {margin-left: 0; width: 100%;}
    button[data-baseweb="tab"] {padding: 5px 12px;}
    .stButton > button, .stDownloadButton > button {width: 100%;}
    .st-key-theme_toggle button {width: 44px;}
}
</style>
""", unsafe_allow_html=True)

TITLES = {
    "missing_values": "Missing values",
    "structural_missing": "Blank by design",
    "disguised_missing": "Placeholder values (like N/A)",
    "exact_duplicates": "Duplicate rows",
    "duplicate_ids": "Repeated IDs",
    "extra_whitespace": "Extra spaces",
    "inconsistent_case": "Inconsistent spelling or case",
    "numeric_as_text": "Numbers stored as text",
    "mixed_types": "Text mixed into numbers",
    "impossible_values": "Impossible values",
    "extreme_outliers": "Extreme outliers",
    "future_dates": "Dates in the future",
    "date_order": "Dates in the wrong order",
    "suspicious_rate": "Suspiciously high rate",
    "constant_column": "Column never changes",
    "all_unique_column": "Every value is unique",
    "messy_column_name": "Messy column name",
    "date_gaps": "Missing days in the data",
    "rare_categories": "Rare values (possible typos)",
}
SEV = {"high": ("hi", "High"), "medium": ("me", "Medium"), "low": ("lo", "Low")}
VERDICT_STYLE = {
    "SAFE TO ANALYZE": "linear-gradient(135deg, #27B5A5, #0F8F97)",
    "USE WITH CAUTION": "linear-gradient(135deg, #F5B54A, #E08A1E)",
    "DO NOT USE UNTIL FIXED": "linear-gradient(135deg, #F0716B, #D6453F)",
}
BAR = {"SAFE TO ANALYZE": "#22B07D", "USE WITH CAUTION": "#E08A1E", "DO NOT USE UNTIL FIXED": "#D6453F"}


def esc(x):
    return html.escape(str(x))


def bold_cols(text, columns):
    """Escape text and make any 'column name' in quotes bold."""
    out = html.escape(str(text))
    for c in sorted(map(str, columns), key=len, reverse=True):
        out = out.replace(html.escape(f"'{c}'"), f"<b>{html.escape(c)}</b>")
    return out


def table_html(frame):
    """A simple styled table where the 'column' column is bold."""
    head = "".join(f"<th>{esc(c)}</th>" for c in frame.columns)
    body = ""
    for _, row in frame.iterrows():
        cells = "".join(
            f'<td class="cn">{esc(v)}</td>' if c == "column" else f"<td>{esc(v)}</td>"
            for c, v in zip(frame.columns, row.values))
        body += f"<tr>{cells}</tr>"
    return f'<div class="tbl"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def title_of(check):
    return TITLES.get(check, check.replace("_", " ").capitalize())


def plain(f):
    """One easy sentence about a finding, with the column and the key words in bold."""
    col = f"<b>{esc(f.column)}</b>"
    pct = f"{f.pct:g}%"
    sentences = {
        "missing_values": f"{col} information is <b>missing</b> in {pct} of rows",
        "structural_missing": f"{col} is <b>blank by design</b> in some rows",
        "disguised_missing": f"{col} has <b>placeholder values</b> (like N/A) in {pct} of rows",
        "exact_duplicates": f"<b>{f.count:,} duplicate rows</b> were found",
        "duplicate_ids": f"{col} has <b>repeated IDs</b>",
        "extra_whitespace": f"{col} has <b>extra spaces</b> in some values",
        "inconsistent_case": f"{col} is <b>spelled inconsistently</b> in some rows",
        "numeric_as_text": f"{col} holds numbers saved as <b>text</b>",
        "mixed_types": f"{col} has <b>text mixed into numbers</b>",
        "impossible_values": f"{col} has <b>impossible values</b> in {pct} of rows",
        "extreme_outliers": f"{col} has <b>extreme outliers</b>",
        "future_dates": f"{col} has <b>dates in the future</b>",
        "date_order": f"{col} has <b>dates in the wrong order</b>",
        "suspicious_rate": f"{col} has a <b>suspiciously high rate</b> ({pct})",
        "constant_column": f"{col} <b>never changes</b>",
        "all_unique_column": f"{col} has <b>no repeated values</b>",
        "messy_column_name": f"{col} has a <b>messy column name</b>",
        "date_gaps": f"{col} has <b>missing days</b>",
        "rare_categories": f"{col} has <b>rare values</b> (possible typos)",
    }
    return sentences.get(f.check, f"<b>{esc(title_of(f.check))}</b> in {col}")


def affected(f):
    return f"{f.count:,} rows ({f.pct}%)" if f.count else "Column-level issue"


def bars(rows, colors):
    """rows: list of (label, value, color). Draws simple horizontal bars."""
    top = max([v for _, v, _ in rows] + [1])
    out = ""
    for label, value, color in rows:
        out += (f'<div class="brow"><div class="blabel">{esc(label)}</div>'
                f'<div class="btrack"><div class="bfill" style="width:{value / top * 100:.0f}%;background:{color};"></div></div>'
                f'<div class="bcount">{value}</div></div>')
    return out


def load_csv(file):
    raw = file.getvalue()
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError("unreadable")


@st.cache_resource
def logo_html(theme):
    """The logo with dark letters for light mode, and light letters for dark mode."""
    def encode(path):
        with open(path, "rb") as fh:
            return base64.b64encode(fh.read()).decode()
    if not os.path.exists("logo_clean.png"):
        return "<h1>CheckKro</h1>"
    path = "logo_dark.png" if theme == "dark" and os.path.exists("logo_dark.png") else "logo_clean.png"
    return f'<img class="logo" src="data:image/png;base64,{encode(path)}" alt="CheckKro">'


@st.cache_data(show_spinner="Checking your file...")
def analyse(df):
    findings = checks.run_checkup(df)
    cleaned, log = cleaner.clean(df)
    after = checks.run_checkup(cleaned)
    return findings, cleaned, log, after


# ---------- header and upload ----------
st.markdown(logo_html(mode), unsafe_allow_html=True)
st.markdown('<div class="tagline">Know if your data is safe to analyze.</div>', unsafe_allow_html=True)

uploaded = st.file_uploader("Upload a CSV file", type="csv")
if uploaded is None:
    if st.button("Try with a sample messy file"):
        st.session_state["sample"] = True
else:
    st.session_state["sample"] = False

df = None
if uploaded is not None:
    try:
        df, enc = load_csv(uploaded)
    except Exception:
        st.error("This file could not be read as a CSV. Check that it is a plain CSV export and try again.")
        st.stop()
    if enc != "utf-8-sig":
        st.info(f"This file is not UTF-8, so it was read as {enc}. Special characters may look different.")
elif st.session_state.get("sample"):
    df = make_messy()
    st.info("Showing the built-in sample file.")

if df is None:
    st.stop()
if df.shape[0] == 0 or df.shape[1] == 0:
    st.warning("This file has no rows to check. Upload a file that contains data.")
    st.stop()

findings, cleaned, log, after = analyse(df)
score = scoring.trust_score(findings)
verdict = scoring.verdict(score, findings)
n_sev = {s: sum(f.severity == s for f in findings) for s in ("high", "medium", "low")}
n_high = n_sev["high"]
affected_cols = Counter(f.column for f in findings if f.column != "(all columns)")

# ---------- dashboard: metrics (the trust score bar is coloured by the result) ----------
bar = BAR[verdict]
if score >= 85 and n_high > 0:
    bar_note = "High score, but a serious issue blocks 'safe'"
elif verdict == "SAFE TO ANALYZE":
    bar_note = "Safe to analyze (85 and above)"
elif verdict == "USE WITH CAUTION":
    bar_note = "Use with caution (60 to 84)"
else:
    bar_note = "Do not use until fixed (below 60)"
c1, c2, c3, c4 = st.columns(4)
c1.markdown(
    f'<div class="scorecard" style="border-top-color:{bar};"><div class="slabel">Trust score</div>'
    f'<div class="svalue">{score} / 100</div>'
    f'<div class="sbar"><div class="sfill" style="width:{max(0, min(score, 100))}%;background:{bar};"></div>'
    f'<div class="stick" style="left:60%;"></div><div class="stick" style="left:85%;"></div></div>'
    f'<div class="snote">{bar_note}</div></div>',
    unsafe_allow_html=True,
)
c2.metric("Problems found", len(findings))
c3.metric("Columns affected", len(affected_cols))
c4.metric("Rows x columns", f"{len(df):,} x {df.shape[1]}")

# ---------- verdict card, explained in plain words ----------
serious = [f for f in findings if f.severity == "high"]
worth_a_look = [f for f in findings if f.severity == "medium"]
if not findings:
    reason = '<div class="rhead">No problems found.</div>'
else:
    if serious:
        head = "1 issue needs your attention" if len(serious) == 1 else f"{len(serious)} issues need your attention"
        shown_items, rest = serious[:4], len(serious) - 4
    elif worth_a_look:
        head = "No serious issues. Worth a look:"
        shown_items, rest = worth_a_look[:3], len(worth_a_look) - 3
    else:
        head, shown_items, rest = "No serious issues. Only minor notes.", [], 0
    reason = f'<div class="rhead">{esc(head)}</div>'
    if shown_items:
        reason += "<ul>" + "".join(f"<li>{plain(f)}</li>" for f in shown_items) + "</ul>"
    if rest > 0:
        reason += f'<div class="rmore">and {rest} more in the Findings tab</div>'
st.markdown(
    f'<div class="verdict-card" style="background:{VERDICT_STYLE[verdict]};">'
    f'<div><div class="label">Verdict</div><div class="big">{esc(verdict)}</div></div>'
    f'<div class="reason">{reason}</div></div>',
    unsafe_allow_html=True,
)

# ---------- dashboard: breakdown panels ----------
left, right = st.columns(2)
with left:
    rows = [("High", n_sev["high"], "#D6453F"), ("Medium", n_sev["medium"], "#E08A1E"),
            ("Low", n_sev["low"], "#8CA0BF")]
    st.markdown(f'<div class="panel"><div class="ptitle">Findings by severity</div>{bars(rows, None)}</div>',
                unsafe_allow_html=True)
with right:
    top_cols = [(c, n, "var(--accent)") for c, n in affected_cols.most_common(5)]
    body = bars(top_cols, None) if top_cols else '<div class="frow">No column has problems.</div>'
    st.markdown(f'<div class="panel"><div class="ptitle">Columns needing attention</div>{body}</div>',
                unsafe_allow_html=True)

tab_findings, tab_score, tab_overview, tab_charts, tab_clean = st.tabs(
    ["Findings", "Score explained", "Overview", "Charts", "Clean the file"])

# ---------- findings ----------
with tab_findings:
    show = st.radio("Show", ["All", "High", "Medium", "Low"], horizontal=True, label_visibility="collapsed")
    shown = [f for f in findings if show == "All" or f.severity == show.lower()]
    if not findings:
        st.success("No problems found.")
    elif not shown:
        st.info("No findings at this severity.")
    for f in shown:
        css, label = SEV[f.severity]
        st.markdown(
            f'<div class="finding"><div class="fhead">'
            f'<span class="pill pill-{css}">{label}</span>'
            f'<span class="ftitle">{esc(title_of(f.check))}</span>'
            f'<span class="fcol">{esc(f.column)}</span>'
            f'<span class="faff">{esc(affected(f))}</span></div>'
            f'<div class="frow"><b>Why it matters:</b> {bold_cols(f.why, df.columns)}</div>'
            f'<div class="frow"><b>Suggested action:</b> {bold_cols(f.suggestion, df.columns)}</div></div>',
            unsafe_allow_html=True,
        )
        with st.expander("Details and impact on your numbers"):
            st.markdown(f'<div class="frow">{bold_cols(f.detail, df.columns)}</div>', unsafe_allow_html=True)
            effect = impact.impact(df, f)
            if effect:
                st.markdown(f'<div class="frow"><b>Impact on your numbers:</b> {bold_cols(effect, df.columns)}</div>',
                            unsafe_allow_html=True)

# ---------- score explained ----------
with tab_score:
    pen = scoring.PENALTY
    st.markdown("**How the score is calculated.** The score starts at 100. Every finding takes points off, "
                "depending on its severity:")
    for s in ("high", "medium", "low"):
        st.markdown(f"- {n_sev[s]} {s} finding(s) x {pen[s]} points = **-{n_sev[s] * pen[s]}**")
    st.markdown(f"Result: **{score} / 100**.")
    costly = sorted(findings, key=lambda f: -pen[f.severity])[:5]
    if costly:
        st.markdown("**What lowers it most:**")
        for f in costly:
            st.markdown(f"- -{pen[f.severity]}: {title_of(f.check)} in **{f.column}**")
    st.markdown("**What the verdict means:**")
    st.markdown("- **Safe to analyze:** score of 85 or more, and no high-severity findings.\n"
                "- **Use with caution:** score of 60 or more.\n"
                "- **Do not use until fixed:** anything below 60.")
    st.caption("The score is a simple rule-based heuristic: fixed penalties by severity. It is not a statistical "
               "probability and does not guarantee that the data is right or wrong. One high-severity finding "
               "always blocks 'safe to analyze'.")

# ---------- overview ----------
with tab_overview:
    ov = checks.overview(df)
    ov["example"] = ov["example"].astype(str)
    st.subheader("Columns")
    st.markdown(table_html(ov), unsafe_allow_html=True)
    cov = checks.date_coverage(df)
    if len(cov) > 0:
        st.subheader("Date coverage")
        st.markdown(table_html(cov), unsafe_allow_html=True)

# ---------- charts ----------
with tab_charts:
    charts.TEAL = "#38BDF8" if mode == "dark" else "#0F9AA8"
    charts.render_charts(df, cleaned)

# ---------- clean ----------
with tab_clean:
    st.write("Only safe fixes are applied: trimming spaces, turning placeholders like 'N/A' "
             "into real blanks, and removing exact duplicate rows. Everything else needs your decision. "
             "Your original file is never changed.")
    placeholders = int(log.loc[log["action"].str.startswith("placeholder"), "rows_affected"].sum()) if len(log) else 0
    after_score = scoring.trust_score(after)
    m1, m2, m3 = st.columns(3)
    m1.metric("Rows", f"{len(df):,} -> {len(cleaned):,}")
    m2.metric("Placeholders turned blank", f"{placeholders:,}")
    m3.metric("Score after safe fixes", f"{after_score} / 100", delta=after_score - score)
    if len(log) == 0:
        st.info("Nothing needed the safe fixes.")
    else:
        st.subheader("Change log")
        st.markdown(table_html(log), unsafe_allow_html=True)
    st.subheader("Preview of the cleaned data")
    preview = cleaned.head(50).astype(object).where(lambda x: x.notna(), "")
    st.markdown(table_html(preview), unsafe_allow_html=True)
    d1, d2 = st.columns(2)
    d1.download_button("Download cleaned CSV", cleaned.to_csv(index=False).encode("utf-8"),
                       file_name="cleaned.csv", mime="text/csv")
    d2.download_button("Download change log (CSV)", log.to_csv(index=False).encode("utf-8"),
                       file_name="change_log.csv", mime="text/csv")

st.caption("CheckKro uses rule-based checks. Nothing in your file is changed without a logged record.")

Overwriting app.py


**Add more chart types.** `charts.py` now offers eleven charts: bar, stacked bar, line, area, pie, donut, scatter, histogram, box plot, correlation heatmap and a missing-values chart. Columns that are mostly numbers saved as text are read as numbers for charting, and the settings panel says which columns were left out of the number pickers and why. If a chart cannot be drawn for a selection, the app shows a short message instead of an error.

In [44]:
%%writefile charts.py
import altair as alt
import numpy as np
import pandas as pd

import checks

alt.data_transformers.disable_max_rows()

TEAL = "#0F9AA8"
NEUTRAL = "#7A879E"   # reads on both light and dark backgrounds
WARM = "#E08A1E"
RED = "#E0584F"
COUNT = "(count of rows)"
PERIOD = {"Day": "D", "Week": "W", "Month": "M"}
KINDS = ["Bar", "Stacked bar", "Line", "Area", "Pie", "Donut", "Scatter", "Histogram",
         "Box plot", "Correlation heatmap", "Missing values"]


def palette():
    return [TEAL, "#38BDF8", "#6366F1", "#F59E0B", "#EC4899", "#10B981", "#94A3B8", "#A78BFA"]


def is_num(s):
    return pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)


def prepare(raw):
    """Chart-ready copy of the data. Text columns that are mostly numbers are read as numbers.
    Returns (data, converted, numeric, dates, why_not)."""
    data = raw.copy()
    dates = checks.date_columns(raw)
    converted, why_not = [], {}
    for c in raw.columns:
        s = raw[c]
        if c in dates:
            why_not[c] = "date"
        elif is_num(s):
            continue
        elif pd.api.types.is_bool_dtype(s):
            why_not[c] = "yes/no"
        elif checks.is_text(s):
            txt = s.astype(str).str.strip().str.replace(",", "", regex=False)
            num = pd.to_numeric(txt, errors="coerce")
            if s.notna().any() and num[s.notna()].notna().mean() >= 0.8:
                data[c] = num
                converted.append(c)
            else:
                why_not[c] = "text"
        else:
            why_not[c] = "other"
    numeric = [c for c in data.columns if is_num(data[c])]
    return data, converted, numeric, dates, why_not


def style(chart, title, height=340):
    return (chart.properties(title=title, height=height, background="transparent")
            .configure_axis(labelColor=NEUTRAL, titleColor=NEUTRAL, gridColor="rgba(128,128,128,0.22)",
                            domainColor="rgba(128,128,128,0.4)", labelFontSize=12, titleFontSize=12)
            .configure_legend(labelColor=NEUTRAL, titleColor=NEUTRAL, labelFontSize=12)
            .configure_title(color=NEUTRAL, fontSize=15, fontWeight=600, anchor="start")
            .configure_view(stroke=None))


def ylabel_of(y, agg):
    return "Number of rows" if y == COUNT else f"{agg} of {y}"


def auto_title(kind, x=None, y=None, agg="sum", split=None):
    if kind in ("Bar", "Line", "Area", "Pie", "Donut"):
        return f"{ylabel_of(y, agg)} by {x}"
    if kind == "Stacked bar":
        return f"{ylabel_of(y, agg)} by {x}, split by {split}"
    if kind == "Scatter":
        return f"{y} vs {x}"
    if kind == "Histogram":
        return f"Distribution of {x}"
    if kind == "Box plot":
        return f"Spread of {y}" + (f" by {x}" if x else "")
    if kind == "Correlation heatmap":
        return "How the number columns move together"
    return "Missing values by column"


def _period_labels(series, freq):
    return series.dt.to_period(PERIOD[freq]).dt.to_timestamp()


def _group(df, label, y, agg):
    if y == COUNT:
        return pd.DataFrame({"label": label}).dropna().groupby("label").size().reset_index(name="value")
    return (pd.DataFrame({"label": label, "v": df[y]}).dropna()
            .groupby("label")["v"].agg(agg).reset_index(name="value"))


def _bar_line_area(df, kind, x, y, agg, top_n, freq, dates, title, sort_by, labels):
    is_date = x in dates
    if is_date:
        label = _period_labels(dates[x], freq)
    else:
        label = df[x] if is_num(df[x]) else df[x].astype(str)
    g = _group(df, label, y, agg)
    ylabel = ylabel_of(y, agg)
    if g.empty:
        return None, None, "There is nothing to plot for this selection."
    if kind == "Bar" and not is_date:
        g = g.sort_values("value", ascending=False).head(top_n)
        g["label"] = g["label"].astype(str)
        if sort_by == "Smallest first":
            g = g.sort_values("value")
        elif sort_by == "A to Z":
            g = g.sort_values("label")
    else:
        g = g.sort_values("label")
    if is_date:
        xenc = alt.X("label:T", title=x)
    elif kind in ("Line", "Area") and is_num(g["label"]):
        xenc = alt.X("label:Q", title=x)
    else:
        xenc = alt.X("label:N", sort=None, title=x)
    base = alt.Chart(g)
    if kind == "Bar":
        mark = base.mark_bar(color=TEAL, cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    elif kind == "Area":
        mark = base.mark_area(color=TEAL, opacity=0.35, line={"color": TEAL})
    else:
        mark = base.mark_line(color=TEAL, strokeWidth=2.5, point=alt.OverlayMarkDef(color=TEAL))
    chart = mark.encode(x=xenc, y=alt.Y("value:Q", title=ylabel),
                        tooltip=[alt.Tooltip("label", title=x), alt.Tooltip("value:Q", title=ylabel, format=",.2f")])
    if labels and len(g) <= 25:
        whole = bool((g["value"] % 1 == 0).all())
        text = base.mark_text(dy=-9, fontSize=11, fontWeight="bold", color=NEUTRAL).encode(
            x=xenc, y=alt.Y("value:Q", title=ylabel),
            text=alt.Text("value:Q", format=",.0f" if whole else ",.1f"))
        chart = chart + text
    note = f"Showing the top {top_n} values." if kind == "Bar" and not is_date and len(g) == top_n else None
    return (style(chart, title or auto_title(kind, x, y, agg)),
            g.rename(columns={"label": x, "value": ylabel}), note)


def _stacked(df, x, split, y, agg, top_n, freq, dates, title, pct):
    if x == split:
        return None, None, "Pick two different columns."
    label = (_period_labels(dates[x], freq).dt.strftime("%Y-%m-%d") if x in dates else df[x].astype(str))
    part = df[split].astype(str)
    keep = part.value_counts().head(7).index
    part = part.where(part.isin(keep), "Other")
    ylabel = ylabel_of(y, agg)
    frame = pd.DataFrame({"label": label, "split": part})
    if y == COUNT:
        g = frame.dropna().groupby(["label", "split"]).size().reset_index(name="value")
    else:
        frame["v"] = df[y]
        g = frame.dropna().groupby(["label", "split"])["v"].agg(agg).reset_index(name="value")
    if g.empty:
        return None, None, "There is nothing to plot for this selection."
    totals = g.groupby("label")["value"].sum().sort_values(ascending=False)
    order = [str(v) for v in (totals.index if x in dates else totals.head(top_n).index)]
    if x in dates:
        order = sorted(order)
    g = g[g["label"].isin(order)]
    chart = (alt.Chart(g).mark_bar()
             .encode(x=alt.X("label:N", sort=order, title=x),
                     y=alt.Y("value:Q", stack="normalize" if pct else "zero", title="Share of each bar" if pct else ylabel,
                             axis=alt.Axis(format="%") if pct else alt.Axis()),
                     color=alt.Color("split:N", scale=alt.Scale(range=palette()), legend=alt.Legend(title=split)),
                     tooltip=[alt.Tooltip("label", title=x), alt.Tooltip("split", title=split),
                              alt.Tooltip("value:Q", title=ylabel, format=",.2f")]))
    notes = []
    if x not in dates and len(totals) > top_n:
        notes.append(f"Showing the top {top_n} groups.")
    if df[split].astype(str).nunique() > 7:
        notes.append(f"The 7 biggest values of '{split}' are shown, and the rest are grouped as Other.")
    note = " ".join(notes) or None
    return (style(chart, title or auto_title("Stacked bar", x, y, agg, split)),
            g.rename(columns={"label": x, "split": split, "value": ylabel}), note)


def _pie(df, kind, x, y, agg, top_n, title):
    g = _group(df, df[x].astype(str), y, agg)
    if g.empty:
        return None, None, "There is nothing to plot for this selection."
    if (g["value"] < 0).any():
        return None, None, "Pie charts need values that are zero or above. Pick another column or use a bar chart."
    g = g.sort_values("value", ascending=False)
    additive = (y == COUNT or agg == "sum")
    note = None
    top = g.head(top_n)
    if len(g) > top_n:
        if additive:
            top = pd.concat([top, pd.DataFrame({"label": ["Other"], "value": [g.iloc[top_n:]["value"].sum()]})])
            note = f"Showing the top {top_n} slices; the rest are grouped as Other."
        else:
            note = f"Showing the top {top_n} slices only."
    total = top["value"].sum()
    if total <= 0:
        return None, None, "All values are zero, so there is nothing to draw."
    top = top.assign(share=top["value"] / total * 100)
    top["legend"] = top.apply(lambda r: f"{r['label']} ({r['share']:.1f}%)", axis=1)
    order = list(top["legend"])
    ylabel = ylabel_of(y, agg)
    chart = (alt.Chart(top).mark_arc(innerRadius=70 if kind == "Donut" else 0, outerRadius=140)
             .encode(theta=alt.Theta("value:Q", stack=True),
                     color=alt.Color("legend:N", sort=order, scale=alt.Scale(range=palette()),
                                     legend=alt.Legend(title=x)),
                     tooltip=[alt.Tooltip("label", title=x), alt.Tooltip("value:Q", title=ylabel, format=",.2f"),
                              alt.Tooltip("share:Q", title="Share (%)", format=".1f")]))
    return (style(chart, title or auto_title(kind, x, y, agg)),
            top[["label", "value", "share"]].rename(columns={"label": x, "value": ylabel, "share": "share_%"}), note)


def _scatter(df, x, y, title):
    if x == y:
        return None, None, "Pick two different columns."
    d = df[[x, y]].dropna().rename(columns={x: "x", y: "y"})
    if d.empty:
        return None, None, "There is nothing to plot for this selection."
    note = None
    if len(d) > 3000:
        note = f"Showing a random sample of 3,000 points out of {len(d):,}."
        d = d.sample(3000, random_state=0)
    chart = (alt.Chart(d).mark_circle(size=45, opacity=0.45, color=TEAL)
             .encode(x=alt.X("x:Q", title=x), y=alt.Y("y:Q", title=y),
                     tooltip=[alt.Tooltip("x:Q", title=x), alt.Tooltip("y:Q", title=y)]))
    return style(chart, title or auto_title("Scatter", x, y)), d.rename(columns={"x": x, "y": y}), note


def _hist(df, x, bins, title):
    s = pd.to_numeric(df[x], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return None, None, "There are no numbers to plot in this column."
    counts, edges = np.histogram(s, bins=bins)
    h = pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})
    chart = (alt.Chart(h).mark_bar(color=TEAL, opacity=0.9)
             .encode(x=alt.X("start:Q", title=x), x2="end:Q", y=alt.Y("count:Q", title="Number of rows"),
                     tooltip=[alt.Tooltip("start:Q", title="From", format=",.2f"),
                              alt.Tooltip("end:Q", title="To", format=",.2f"),
                              alt.Tooltip("count:Q", title="Rows")]))
    return style(chart, title or auto_title("Histogram", x)), h, None


def _box(df, y, group, title):
    vals = pd.to_numeric(df[y], errors="coerce").replace([np.inf, -np.inf], np.nan)
    if group:
        keys = df[group].astype(str)
        top = keys.value_counts().head(8).index
        parts = {k: vals[(keys == k)].dropna() for k in top}
    else:
        parts = {"All rows": vals.dropna()}
    stats, outs = [], []
    for name, v in parts.items():
        if len(v) == 0:
            continue
        q1, med, q3 = np.percentile(v, [25, 50, 75])
        iqr = q3 - q1
        lo_f, hi_f = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        inside = v[(v >= lo_f) & (v <= hi_f)]
        low = inside.min() if len(inside) else q1
        high = inside.max() if len(inside) else q3
        out = v[(v < lo_f) | (v > hi_f)]
        stats.append({"group": name, "low": low, "q1": q1, "median": med, "q3": q3, "high": high,
                      "rows": len(v), "outliers": len(out)})
        outs.append(pd.DataFrame({"group": name, "value": out.head(200).values}))
    if not stats:
        return None, None, "There are no numbers to plot in this column."
    st_df = pd.DataFrame(stats)
    xenc = alt.X("group:N", sort=list(st_df["group"]), title=group or None)
    rule = alt.Chart(st_df).mark_rule(color=NEUTRAL).encode(x=xenc, y=alt.Y("low:Q", title=y), y2="high:Q")
    box = (alt.Chart(st_df).mark_bar(color=TEAL, size=44, opacity=0.9)
           .encode(x=xenc, y="q1:Q", y2="q3:Q",
                   tooltip=[alt.Tooltip("group", title="Group"), alt.Tooltip("median:Q", format=",.2f"),
                            alt.Tooltip("q1:Q", title="25th percentile", format=",.2f"),
                            alt.Tooltip("q3:Q", title="75th percentile", format=",.2f"),
                            alt.Tooltip("rows:Q", title="Rows"), alt.Tooltip("outliers:Q", title="Outliers")]))
    med = alt.Chart(st_df).mark_tick(color="white", size=44, thickness=2).encode(x=xenc, y="median:Q")
    chart = rule + box + med
    out_df = pd.concat(outs, ignore_index=True)
    note = None
    if len(out_df):
        pts = alt.Chart(out_df).mark_circle(color=WARM, size=35, opacity=0.7).encode(
            x=alt.X("group:N", sort=list(st_df["group"])), y="value:Q",
            tooltip=[alt.Tooltip("value:Q", title="Outlier", format=",.2f")])
        chart = chart + pts
        note = "Orange dots are outliers (more than 1.5 times the middle spread beyond the box). Up to 200 are drawn per group."
    return style(chart, title or auto_title("Box plot", group, y)), st_df, note


def _heatmap(df, numeric, title):
    cols = [c for c in numeric if df[c].nunique() > 1][:15]
    if len(cols) < 2:
        return None, None, "A heatmap needs at least two number columns that are not constant."
    corr = df[cols].corr(numeric_only=True)
    long = corr.stack().reset_index()
    long.columns = ["x", "y", "value"]
    long = long.dropna()
    long["value"] = long["value"].round(2)
    long["strong"] = long["value"].abs() > 0.6
    xenc = alt.X("x:N", sort=cols, title=None, axis=alt.Axis(labelAngle=-45))
    yenc = alt.Y("y:N", sort=cols, title=None)
    rect = (alt.Chart(long).mark_rect()
            .encode(x=xenc, y=yenc,
                    color=alt.Color("value:Q", scale=alt.Scale(domain=[-1, 0, 1], range=[RED, "#CBD5E1", TEAL]),
                                    legend=alt.Legend(title="Correlation")),
                    tooltip=[alt.Tooltip("x", title="Column"), alt.Tooltip("y", title="Compared with"),
                             alt.Tooltip("value:Q", title="Correlation", format=".2f")]))
    text = (alt.Chart(long).mark_text(fontSize=11)
            .encode(x=xenc, y=yenc, text=alt.Text("value:Q", format=".2f"),
                    color=alt.condition("datum.strong", alt.value("white"), alt.value("#0B1B3A"))))
    note = "Correlation shows how two columns move together, from -1 to 1. It does not mean one causes the other."
    if len([c for c in numeric if df[c].nunique() > 1]) > 15:
        note = "Showing the first 15 number columns. " + note
    return style(rect + text, title or auto_title("Correlation heatmap"), height=420), corr.reset_index(), note


def _missing(df, title):
    ov = checks.overview(df)
    d = (ov[ov["missing_%"] > 0][["column", "missing_%"]]
         .sort_values("missing_%", ascending=False).head(25).rename(columns={"missing_%": "pct"}))
    if d.empty:
        return None, None, "No missing values found in this file."
    bars = (alt.Chart(d).mark_bar(cornerRadiusEnd=4)
            .encode(y=alt.Y("column:N", sort="-x", title=None), x=alt.X("pct:Q", title="% of rows missing"),
                    color=alt.condition("datum.pct > 20", alt.value(RED), alt.value(TEAL)),
                    tooltip=[alt.Tooltip("column", title="Column"), alt.Tooltip("pct:Q", title="% missing", format=".1f")]))
    text = (alt.Chart(d).mark_text(align="left", dx=4, fontSize=11, fontWeight="bold", color=NEUTRAL)
            .encode(y=alt.Y("column:N", sort="-x"), x="pct:Q", text=alt.Text("pct:Q", format=".1f")))
    note = "Counts blank cells and placeholders such as N/A. Bars above 20% are shown in red."
    return (style(bars + text, title or auto_title("Missing values"), height=max(120, len(d) * 28 + 40)),
            d.rename(columns={"pct": "missing_%"}), note)


def build_chart(df, kind, x=None, y=None, agg="sum", top_n=10, bins=30, freq="Day", dates=None,
                title=None, sort_by="Largest first", labels=True, split=None, pct=False, numeric=None):
    """Returns (chart, plotted_data, note). chart is None when nothing can be drawn."""
    dates = dates or {}
    if kind in ("Bar", "Line", "Area"):
        return _bar_line_area(df, kind, x, y, agg, top_n, freq, dates, title, sort_by, labels)
    if kind == "Stacked bar":
        return _stacked(df, x, split, y, agg, top_n, freq, dates, title, pct)
    if kind in ("Pie", "Donut"):
        return _pie(df, kind, x, y, agg, top_n, title)
    if kind == "Scatter":
        return _scatter(df, x, y, title)
    if kind == "Histogram":
        return _hist(df, x, bins, title)
    if kind == "Box plot":
        return _box(df, y, x, title)
    if kind == "Correlation heatmap":
        return _heatmap(df, numeric or [c for c in df.columns if is_num(df[c])], title)
    return _missing(df, title)


def hidden_note(cols, numeric, why_not):
    left_out = [f"{c} ({why_not.get(c, 'other')})" for c in cols if c not in numeric]
    if not left_out:
        return None
    shown = ", ".join(left_out[:6]) + (f" and {len(left_out) - 6} more" if len(left_out) > 6 else "")
    return f"Not offered here, because they are not numbers: {shown}."


def render_charts(df, cleaned):
    import streamlit as st

    left, right = st.columns([1, 2.2])
    problem = None
    chart = plotted = note = None

    with left:
        st.markdown("**Chart settings**")
        source = st.radio("Data", ["Original", "Cleaned"], horizontal=True, key="ch_source")
        data, converted, numeric, dates, why_not = prepare(df if source == "Original" else cleaned)
        cols = list(data.columns)
        kind = st.selectbox("Chart type", KINDS, key="ch_kind")

        x = y = split = None
        agg, top_n, bins, freq, pct = "sum", 10, 30, "Day", False
        sort_by, labels = "Largest first", True
        guess = next((i for i, c in enumerate(cols)
                      if c not in numeric and c not in dates and 2 <= data[c].nunique() <= 50), 0)
        needs_numbers = False

        if kind in ("Bar", "Line", "Area", "Stacked bar", "Pie", "Donut"):
            xlabel = {"Pie": "Slice by column", "Donut": "Slice by column"}.get(kind, "Group by column")
            x = st.selectbox(xlabel, cols, index=guess, key="ch_x")
            if kind == "Stacked bar":
                options = [c for c in cols if c != x]
                if not options:
                    problem = "A stacked bar needs at least two columns."
                else:
                    pick = next((i for i, c in enumerate(options) if c not in numeric and 2 <= data[c].nunique() <= 20), 0)
                    split = st.selectbox("Split each bar by", options, index=pick, key="ch_split")
            y = st.selectbox("Y value", [COUNT] + numeric, key="ch_y")
            needs_numbers = True
            if y != COUNT:
                agg = st.selectbox("Combine values by", ["sum", "mean", "median", "max", "min"], key="ch_agg")
            if x in dates and kind in ("Bar", "Line", "Area", "Stacked bar"):
                span = (dates[x].max() - dates[x].min()).days if dates[x].notna().any() else 0
                default = 2 if span > 180 else 1 if span > 45 else 0
                freq = st.selectbox("Group dates by", ["Day", "Week", "Month"], index=default, key="ch_freq")
            elif kind in ("Bar", "Stacked bar"):
                top_n = st.slider("Top groups", 3, 30, 10, key="ch_top")
            if kind in ("Pie", "Donut"):
                top_n = st.slider("Top slices", 3, 7, 6, key="ch_slices")
            if kind == "Bar" and x not in dates:
                sort_by = st.selectbox("Order bars", ["Largest first", "Smallest first", "A to Z"], key="ch_sort")
            if kind == "Stacked bar":
                pct = st.checkbox("Show as % of each bar", value=False, key="ch_pct")
            if kind in ("Bar", "Line", "Area"):
                labels = st.checkbox("Show value labels", value=True, key="ch_labels")
        elif kind == "Scatter":
            needs_numbers = True
            if len(numeric) < 2:
                problem = "A scatter chart needs at least two number columns."
            else:
                x = st.selectbox("X axis", numeric, key="ch_sx")
                y = st.selectbox("Y axis", numeric, index=1, key="ch_sy")
        elif kind == "Histogram":
            needs_numbers = True
            if not numeric:
                problem = "A histogram needs a number column."
            else:
                x = st.selectbox("Column", numeric, key="ch_hx")
                bins = st.slider("Number of bars", 10, 60, 30, key="ch_bins")
        elif kind == "Box plot":
            needs_numbers = True
            if not numeric:
                problem = "A box plot needs a number column."
            else:
                y = st.selectbox("Column to measure", numeric, key="ch_by")
                grp = st.selectbox("Compare by (optional)", ["(none)"] + [c for c in cols if c != y], key="ch_bg")
                x = None if grp == "(none)" else grp
        elif kind == "Correlation heatmap":
            if len(numeric) < 2:
                problem = "A heatmap needs at least two number columns."
        else:
            pass

        if converted:
            st.caption("Read as numbers, because most of their values are numbers saved as text: "
                       + ", ".join(converted[:6]) + ("..." if len(converted) > 6 else "") + ".")
        if needs_numbers:
            hint = hidden_note(cols, numeric, why_not)
            if hint:
                st.caption(hint)
        if problem is None:
            title = st.text_input("Chart title", auto_title(kind, x, y, agg, split),
                                  key=f"ch_title_{kind}_{x}_{y}_{agg}_{split}")

    with right:
        if problem is None:
            try:
                chart, plotted, note = build_chart(data, kind, x, y, agg, top_n, bins, freq, dates,
                                                   title=title, sort_by=sort_by, labels=labels,
                                                   split=split, pct=pct, numeric=numeric)
            except Exception as err:
                chart, note = None, f"This chart could not be built for this selection ({type(err).__name__})."
            if chart is None:
                problem = note
        if problem is not None:
            st.info(problem)
        else:
            try:
                st.altair_chart(chart, use_container_width=True, theme=None)
            except Exception as err:
                st.warning(f"This chart could not be drawn for this selection ({type(err).__name__}). "
                           "Try different columns or another chart type.")
            else:
                if note:
                    st.caption(note)
                st.caption(f"This chart uses the {source.lower()} file.")
                st.download_button("Download chart data (CSV)", plotted.to_csv(index=False).encode("utf-8"),
                                   file_name="chart_data.csv", mime="text/csv", key="ch_dl")

Overwriting charts.py


**Fix the score bar and the group sliders.** This cell edits `app.py` and `charts.py` in place. It removes the small tick marks from the trust score bar, and it makes the "Top groups" and "Top slices" sliders stop at the number of different values the chosen column really has. When a column has only a few values, the slider is replaced by a note saying all of them are shown.

In [45]:
def patch(path, edits):
    text = open(path).read()
    report = []
    for old, new, label, *skip in edits:
        if skip and skip[0] in text:
            report.append(f"already done: {label}")
        elif old in text:
            text = text.replace(old, new, 1)
            report.append(f"changed: {label}")
        elif new and new in text:
            report.append(f"already done: {label}")
        elif not new:
            report.append(f"nothing to remove: {label}")
        else:
            report.append(f"NOT FOUND: {label}")
    open(path, "w").write(text)
    return report


HELPER = '''def pick_count(st, label, n_available, cap, default, key):
    """A slider whose maximum follows the number of groups the chosen column really has."""
    n_available = int(n_available)
    top = min(cap, n_available)
    if top <= 3:
        st.caption(f"This column has {n_available} different values, so all of them are shown.")
        return max(n_available, 1)
    return st.slider(label, 3, top, min(default, top), key=f"{key}_{top}")


def render_charts(df, cleaned):'''

charts_edits = [
    ("def render_charts(df, cleaned):", HELPER, "helper that sizes the slider", "def pick_count"),
    ('top_n = st.slider("Top groups", 3, 30, 10, key="ch_top")',
     'top_n = pick_count(st, "Top groups", data[x].nunique(), 30, 10, "ch_top")',
     "Top groups slider"),
    ('top_n = st.slider("Top slices", 3, 7, 6, key="ch_slices")',
     'top_n = pick_count(st, "Top slices", data[x].nunique(), 7, 6, "ch_slices")',
     "Top slices slider"),
]
app_edits = [
    ("""    f'<div class="sbar"><div class="sfill" style="width:{max(0, min(score, 100))}%;background:{bar};"></div>'
    f'<div class="stick" style="left:60%;"></div><div class="stick" style="left:85%;"></div></div>'""",
     """    f'<div class="sbar"><div class="sfill" style="width:{max(0, min(score, 100))}%;background:{bar};"></div></div>'""",
     "tick marks on the score bar"),
    (".stick {position: absolute; top: -3px; bottom: -3px; width: 2px; background: var(--muted); opacity: 0.55;}\n", "",
     "unused tick style"),
]

for line in patch("charts.py", charts_edits) + patch("app.py", app_edits):
    print(line)

changed: helper that sizes the slider
changed: Top groups slider
changed: Top slices slider
changed: tick marks on the score bar
changed: unused tick style


**Prepare for publishing.** This writes `requirements.txt`, the list of libraries the hosting service must install (Streamlit, pandas, numpy and the charting library), and `.gitignore`, which keeps temporary files such as caches and notebook checkpoints out of the GitHub repository.

In [46]:
with open("requirements.txt", "w") as f:
    f.write("streamlit>=1.40\npandas>=2.0\nnumpy>=1.24\naltair\n")

with open(".gitignore", "w") as f:
    f.write("__pycache__/\n.ipynb_checkpoints/\n.DS_Store\n")

print("requirements.txt:")
print(open("requirements.txt").read())
print(".gitignore:")
print(open(".gitignore").read())

requirements.txt:
streamlit>=1.40
pandas>=2.0
numpy>=1.24
altair

.gitignore:
__pycache__/
.ipynb_checkpoints/
.DS_Store



**Write the README.** This saves the project description, features, test results and known limits as `README.md`, the first page anyone sees on GitHub.

In [47]:
%%writefile README.md
# CheckKro

**Know if your data is safe to analyze.**

CheckKro checks any CSV file the way an analyst would on first opening it, and answers one question: can I trust this data? It gives a trust score out of 100, a plain-language verdict, and what each problem does to your numbers.

Live demo: (add the link after deploying)
Code: (add the GitHub link)

## Screenshots

(Add two or three images here after deploying: the verdict card, the Findings tab and the Charts tab, in light and dark mode.)

## What it does

1. **Checks** the file for the problems analysts look for first. There are 19 kinds: missing and hidden-missing values, duplicate rows and IDs, formatting mistakes, impossible values, extreme outliers, date errors, suspicious rates, and more.
2. **Gives a verdict in plain words.** A trust score with a bar that changes color (green, amber or red), and a verdict card such as "1 issue needs your attention: Vehicle Images has placeholder values in 100% of rows".
3. **Explains the impact on your numbers.** Not just "150 duplicate rows", but "total order amount is overstated by 3.1%".
4. **Cleans safely.** It only trims spaces, turns placeholders like "N/A" and "#NAME?" into real blanks, and removes exact duplicates. Every change is in a change log you can download. Your original file is never changed.
5. **Draws quick charts.** Eleven types: bar, stacked bar, line, area, pie, donut, scatter, histogram, box plot, correlation heatmap and a missing-values chart, on the original or the cleaned data.

It works on desktop and phone, in light or dark mode (a button in the top right corner switches them).

## What makes it different

- **Impact, not just detection.** Each finding says what it does to your results.
- **It understands blanks that are by design.** A cancelled ride has no rating, and that is not an error. CheckKro spots blanks that follow a rule in another column and rates them low instead of raising an alarm.
- **Suspicious-data detection.** It flags rates that look too high to be real, such as a 42% refund rate.
- **No silent changes.** The cleaner never guesses, and every change is logged.
- **Rule-based, not AI.** The checks are plain Python, so every result is repeatable and explainable.

## How the trust score works

The score starts at 100. Each finding takes points off by severity: 12 for high, 5 for medium and 1 for low. The verdict is "safe to analyze" at 85 or above with no high-severity finding, "use with caution" at 60 or above, and "do not use until fixed" below that.

The score is a simple heuristic, not a statistical probability. The "Score explained" tab in the app shows exactly how each result was reached.

## How well does it work?

**Planted-error test.** I planted 17 kinds of known error, one at a time, in 3 datasets of different sizes (51 tests in total). CheckKro caught 51 of 51, with 0 serious false alarms on the clean datasets. These errors were written by me on synthetic data, so this shows each check does what it was built to do, not how it performs on every real file.

**Real-file test.** I then ran it on a real dataset of 103,024 ride bookings. The first version scored it 0 out of 100 with 9 high-severity alarms, and most of them were wrong:
- Columns such as ratings and payment method are blank for cancelled rides. That is by design.
- Customer ID repeats because customers book more than once.
- A column full of Excel `#NAME?` errors was the one real problem, and it was under-reported.

I fixed all three, and re-ran the planted-error test to confirm nothing broke. The same file now scores 72 out of 100 with one high-severity finding: the `#NAME?` column, which is a real problem.

## Known limits

- Outliers are found with a simple statistical rule, so naturally skewed data (such as booking values with a long tail) can be flagged even when the values are real.
- Some checks use column names to decide what a column means (for example "amount" or "age"), so they work best with English column names.
- The thresholds, such as a refund rate above 25%, are sensible defaults, not universal truths.
- Pandas turns text like `N/A`, `NA`, `null` and `None` into real blanks while reading a file, so those appear as plain missing values. Placeholders such as `-`, `?`, `unknown` and `#NAME?` stay as text and are caught as placeholders.
- It reads CSV files only.
- On the public demo, your file is processed in memory for your session and is not saved by the app. Still, do not upload confidential data to it. You can run it locally instead.

## Run it yourself

    pip install -r requirements.txt
    streamlit run app.py

Click "Try with a sample messy file" to see it work without uploading anything.

## Files

- `app.py` is the Streamlit app: the dashboard, tabs, light and dark mode.
- `checks.py` holds all the checks and the `run_checkup` function.
- `scoring.py` turns findings into a trust score and a verdict.
- `cleaner.py` applies the safe fixes and writes the change log.
- `impact.py` writes the plain-language impact of each finding.
- `charts.py` builds the eleven chart types.
- `datagen.py` builds the clean and messy test data.
- `evaluate.py` plants errors and measures how many are caught.
- `BuildIt.ipynb` is the notebook the project was built in, step by step.

## Built with

Python, pandas, NumPy, Streamlit and Altair.

## Author

Disha Jaiswal, data analyst.

Overwriting README.md
